In [ ]:
# ============================================================================
# VG360 · YOLOX -> 2D array-grammar safety layer · SINGLE COLAB CELL
#   Formal : Lean 4 kernel-only proofs (Nat/omega, no Mathlib), compiled if lean found
#   Metric : MCC (primary) + Mondrian split-conformal (SCCP) for PPE abstention
#   Train  : SAM (sharpness-aware) wrapping AdamW for the only learned piece (PPE fusion)
#   Serve  : OpenAI-compatible /v1/chat/completions (FastAPI, uvicorn in own loop thread)
# Rules    : R1 forklift line-of-fire (homography grid + 4-connected occlusion ray)
#            R2 static red-zone intrusion
#            R3 barricade perimeter closure (post-chain grammar + 4-connected flood fill)
#            R4 PPE part-whole  PERSON -> HEAD . TORSO . LEGS  (3-valued: OK/VIOLATION/UNKNOWN)
# All alarms pass through a leaky-bucket hysteresis whose anti-flicker bound is proved in Lean.
# Data     : simulated site with a real pinhole camera + YOLOX-like noise (dropouts, steam,
#            truncation, low-res far workers, FPs). Swap `sim_episode` for real YOLOX output via
#            `yolox_to_dets` — the engine only consumes [{cls, conf, box:[x1,y1,x2,y2]}].
# ============================================================================
import os, sys, json, math, time, shutil, threading, asyncio, subprocess, collections, random
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'fastapi', 'uvicorn', 'httpx',
                    'opencv-python-headless'], check=False)
import numpy as np, cv2, torch, torch.nn as nn

SEED = 7
np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED)
RNG = np.random.default_rng(SEED)
T0 = time.time()

def sanitize(o):
    if isinstance(o, dict): return {str(k): sanitize(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [sanitize(v) for v in o]
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): o = float(o)
    if isinstance(o, np.ndarray): return sanitize(o.tolist())
    if torch.is_tensor(o): return sanitize(o.detach().cpu().numpy())
    if isinstance(o, float) and not math.isfinite(o): return None
    return o

# ---------------------------------------------------------------- 1. site + camera
FLOOR_W, FLOOR_H, CELL = 20.0, 15.0, 0.5
GW, GH = int(FLOOR_W / CELL), int(FLOOR_H / CELL)            # 40 x 30 grid
IMG_W, IMG_H, FOCAL, FPS = 640, 480, 420.0, 10
TOK = dict(FREE=0, RED_ZONE=1, BARRIER=2, HAZARD=3, WORKER=4, FORKLIFT=5)
R_LOF = 3.0                     # m, forklift line-of-fire radius
LINK_M = 1.25                   # m, max post-to-post chain link (nominal 1.0 m; corner-skip = 1.41 m must NOT link)
S_MAX, HI, LO = 6, 3, 0         # leaky bucket; Lean: OFF->ON and ON->OFF each need >= HI-LO frames
POST_WIN, POST_MIN = 30, 3      # post persistence: a post cluster must be seen in >= 3 of the last 30 frames
POST_R = 0.35                   # m, clustering radius for post sightings (continuous, not cell-snapped)
N_CAL_FORMAL_BOUND, ALPHA = 29, 0.10
GUARDRAIL = [((2.0, 6.25), (9.0, 6.25)), ((11.0, 6.25), (18.0, 6.25))]   # gap = pedestrian crossing
CROSSING = (10.0, 6.25)
RED_RECT = (14.0, 9.0, 17.0, 12.0)
SPILL_C, RING_HALF = (5.0, 10.0), 2.0
LANE_Y = (2.0, 5.0)

def _look_at(cam, tgt):
    f = tgt - cam; f = f / np.linalg.norm(f)
    r = np.cross(f, [0, 0, 1.0]); r /= np.linalg.norm(r)
    u = np.cross(r, f)
    return np.stack([r, -u, f])
# Camera profile. 'wide_2.8mm' is a typical CCTV lens: strong barrel distortion, floor spans the full width.
CAMERA_PROFILE = 'wide_2.8mm'          # or 'pinhole' (the earlier, distortion-free camera)
if CAMERA_PROFILE == 'wide_2.8mm':
    FOCAL, CAM_C, CAM_TGT = 230.0, np.array([10.0, -2.0, 7.5]), np.array([10.0, 6.5, 0.0])
    DIST_TRUE = np.array([-0.32, 0.10, 0.0005, -0.0004, 0.0])
else:
    FOCAL, CAM_C, CAM_TGT = 420.0, np.array([10.0, -12.0, 9.0]), np.array([10.0, 7.5, 0.0])
    DIST_TRUE = np.zeros(5)
K_CAM = np.array([[FOCAL, 0, IMG_W / 2], [0, FOCAL, IMG_H / 2], [0, 0, 1.0]])
R_NOM = _look_at(CAM_C, CAM_TGT)
CAM_POSE = dict(R=R_NOM.copy(), T=-R_NOM @ CAM_C)      # the TRUE pose; the simulator may bump it
def set_pose(tilt_deg=0.0, pan_deg=0.0):
    a, b = math.radians(tilt_deg), math.radians(pan_deg)
    Rx = np.array([[1, 0, 0], [0, math.cos(a), -math.sin(a)], [0, math.sin(a), math.cos(a)]])
    Ry = np.array([[math.cos(b), 0, math.sin(b)], [0, 1, 0], [-math.sin(b), 0, math.cos(b)]])
    R = Ry @ Rx @ R_NOM; CAM_POSE.update(R=R, T=-R @ CAM_C)
def project(P):                                        # world -> distorted pixels (the real camera)
    P = np.atleast_2d(np.asarray(P, float)); Pc = P @ CAM_POSE['R'].T + CAM_POSE['T']
    uv, _ = cv2.projectPoints(Pc.reshape(-1, 1, 3), np.zeros(3), np.zeros(3), K_CAM, DIST_TRUE)
    return uv.reshape(-1, 2)

# ---------------------------------------------------------------- 1b. lens: iterative undistortion + board-free calibration
def und_pts(uv, K, D):
    """Ideal pixels of observed ones: Newton's method on OpenCV's forward model (k1 k2 p1 p2 k3), per point,
    stopping below 1e-9 px. The fixed-point iteration in OpenCV 4's undistortPoints(Iter) oscillates near the
    corners of a strong barrel lens; this converges to the true inverse wherever the model is one-to-one
    (lens_monotone), identically in the Rust port (tamserve-boxes src/lens.rs)."""
    uv = np.asarray(uv, np.float64).reshape(-1, 2)
    if D is None or not np.any(D): return uv.copy()
    fx, fy, cx, cy = K[0, 0], K[1, 1], K[0, 2], K[1, 2]; k1, k2, p1, p2, k3 = (float(v) for v in D[:5])
    xd, yd = (uv[:, 0] - cx) / fx, (uv[:, 1] - cy) / fy
    x, y = xd.copy(), yd.copy(); live = np.ones(len(x), bool)
    for _ in range(50):
        idx = np.flatnonzero(live)
        if len(idx) == 0: break
        xs, ys = x[idx], y[idx]
        r2 = xs * xs + ys * ys
        cd = 1.0 + k1 * r2 + k2 * r2 * r2 + k3 * r2 * r2 * r2
        dc = k1 + 2.0 * k2 * r2 + 3.0 * k3 * r2 * r2
        ex = xs * cd + 2.0 * p1 * xs * ys + p2 * (r2 + 2.0 * xs * xs) - xd[idx]
        ey = ys * cd + p1 * (r2 + 2.0 * ys * ys) + 2.0 * p2 * xs * ys - yd[idx]
        tx, ty = ex * fx, ey * fy
        done = np.sqrt(tx * tx + ty * ty) < 1e-9
        a = cd + 2.0 * xs * xs * dc + 2.0 * p1 * ys + 6.0 * p2 * xs
        b = 2.0 * xs * ys * dc + 2.0 * p1 * xs + 2.0 * p2 * ys
        d = cd + 2.0 * ys * ys * dc + 6.0 * p1 * ys + 2.0 * p2 * xs
        det = a * d - b * b
        x[idx] = np.where(done, xs, xs - (d * ex - b * ey) / det)
        y[idx] = np.where(done, ys, ys - (a * ey - b * ex) / det)
        live[idx[done]] = False
    return np.c_[fx * x + cx, fy * y + cy]
def und_pts_opencv(uv, K, D):   # cross-check only
    crit = (cv2.TERM_CRITERIA_COUNT | cv2.TERM_CRITERIA_EPS, 100, 1e-9); uv = np.asarray(uv, np.float64).reshape(-1, 1, 2)
    if hasattr(cv2, 'undistortPointsIter'): return cv2.undistortPointsIter(uv, K, D, R=None, P=K, criteria=crit).reshape(-1, 2)
    return cv2.undistortPoints(uv, K, D, R=None, P=K, criteria=crit).reshape(-1, 2)
def apply_h(H, xy):
    xy = np.asarray(xy, np.float64).reshape(-1, 2); q = np.c_[xy, np.ones(len(xy))] @ H.T
    return q[:, :2] / q[:, 2:3]
def lens_monotone(K, D, w=IMG_W, h=IMG_H):
    """r_d(r) must increase over the whole image radius, else undistortion folds back near the edges."""
    r = np.linspace(0, 1.3 * math.hypot(w / 2, h / 2) / K[0, 0], 400); k1, k2, k3 = D[0], D[1], (D[4] if len(D) > 4 else 0)
    return bool(np.all(np.diff(r * (1 + k1 * r**2 + k2 * r**4 + k3 * r**6)) > 0))

from scipy.optimize import least_squares
F_SPEC = FOCAL * 1.05                                   # focal length from the lens datasheet: 5 % off on purpose
K_SPEC = np.array([[F_SPEC, 0, IMG_W / 2], [0, F_SPEC, IMG_H / 2], [0, 0, 1.0]])
MARKERS = np.array([[x, y] for x in (0, 5, 10, 15, 20) for y in (0, 7.5, 15)], float)   # 15 floor markings, edges included
_img_m = project(np.c_[MARKERS, np.zeros(len(MARKERS))]) + RNG.normal(0, 1.0, (len(MARKERS), 2))  # installer clicks, 1 px
FLOOR_LINES = [np.c_[np.linspace(0, 20, 25), np.full(25, y)] for y in (1, 3, 5, 7, 9, 11, 13)] + \
              [np.c_[np.full(25, x), np.linspace(0, 15, 25)] for x in (2, 6, 10, 14, 18)]   # paint, grating edges, rails
_line_obs = [project(np.c_[L, np.zeros(len(L))]) + RNG.normal(0, 0.7, (len(L), 2)) for L in FLOOR_LINES]

def fit_homography(img_pts, K, D):
    H, _ = cv2.findHomography(und_pts(img_pts, K, D), MARKERS, 0); return H
def _robust_fit(res_fn, K):
    """Coarse grid for a start, then bounded least squares; a model that folds inside the image is penalised,
    so the optimiser never settles on one (and lens_monotone re-checks the result anyway)."""
    def res(k):
        D = np.array([k[0], k[1], 0, 0, 0]); r = res_fn(D)
        return r if lens_monotone(K, D) else r * 0 + 1e3
    grid = [(k1, k2) for k1 in np.linspace(-0.6, 0.1, 15) for k2 in np.linspace(-0.1, 0.4, 11)]
    start = min(grid, key=lambda k: float(np.sum(res(k) ** 2)))
    k = least_squares(res, start, bounds=([-1.0, -0.5], [0.5, 0.8])).x
    return np.array([k[0], k[1], 0, 0, 0])
def calib_plumb_line(obs, K):
    """Board-free: choose k1,k2 that make straight floor lines straight again."""
    def res(D):
        out = []
        for o in obs:
            u = und_pts(o, K, D); c = u - u.mean(0); out.append(c @ np.linalg.svd(c)[2][-1])
        return np.concatenate(out)
    return _robust_fit(res, K)
def calib_joint(img_pts, K):
    """Board-free: fit k1,k2 together with the homography so the clicked markers land on their floor positions."""
    def res(D):
        H = fit_homography(img_pts, K, D); return (apply_h(H, und_pts(img_pts, K, D)) - MARKERS).ravel()
    return _robust_fit(res, K)

_G = np.array([[x, y] for x in np.linspace(0.25, 19.75, 40) for y in np.linspace(0.25, 14.75, 30)])
_G_PX = project(np.c_[_G, np.zeros(len(_G))]); _G_R = np.hypot(_G_PX[:, 0] - IMG_W / 2, _G_PX[:, 1] - IMG_H / 2) / math.hypot(IMG_W / 2, IMG_H / 2)
def floor_error(K, D, H):
    e = np.linalg.norm(apply_h(H, und_pts(_G_PX, K, D)) - _G, axis=1)
    return dict(mean_cm=round(100 * e.mean(), 1), edge_cm=round(100 * e[_G_R > 0.7].mean(), 1) if np.any(_G_R > 0.7) else None,
                max_cm=round(100 * e.max(), 1))
LENS_REPORT = {}
_cands = {'none': np.zeros(5), 'plumb_line': calib_plumb_line(_line_obs, K_SPEC), 'joint_markers': calib_joint(_img_m, K_SPEC)}
for name, D in _cands.items():
    ok = lens_monotone(K_SPEC, D)
    LENS_REPORT[name] = dict(k1=round(float(D[0]), 4), k2=round(float(D[1]), 4), monotone=ok,
                             **floor_error(K_SPEC, D, fit_homography(_img_m, K_SPEC, D)))
LENS_METHOD = 'plumb_line'    # ~300 line points: steadier than fitting the lens to 15 clicked markers
D_CAL = _cands[LENS_METHOD]
if not lens_monotone(K_SPEC, D_CAL):
    print(f"[lens] {LENS_METHOD} model folds inside the image: refusing it, falling back to no undistortion"); D_CAL = np.zeros(5)
CALIB = dict(K=K_SPEC, D=D_CAL, H=fit_homography(_img_m, K_SPEC, D_CAL), method=LENS_METHOD)
def img2floor(uv, calib=None):
    c = calib or CALIB; return apply_h(c['H'], und_pts(uv, c['K'], c['D']))
HOMO_ERR_M = LENS_REPORT[LENS_METHOD]['mean_cm'] / 100
if np.any(DIST_TRUE):
    _n, _o = und_pts(_G_PX, K_CAM, DIST_TRUE), und_pts_opencv(_G_PX, K_CAM, DIST_TRUE)
    LENS_REPORT['newton_vs_opencv_floor_max_px'] = float(np.abs(_n - _o).max())
    _corner = np.array([[3.0, 4.0], [636.0, 477.0]]); _cn = und_pts(_corner, K_CAM, DIST_TRUE)
    _fx, _fy, _cx, _cy = K_CAM[0, 0], K_CAM[1, 1], K_CAM[0, 2], K_CAM[1, 2]
    _rt, _ = cv2.projectPoints(np.c_[(_cn[:, 0] - _cx) / _fx, (_cn[:, 1] - _cy) / _fy, np.ones(2)].reshape(-1, 1, 3), np.zeros(3), np.zeros(3), K_CAM, DIST_TRUE)
    LENS_REPORT['newton_corner_roundtrip_px'] = float(np.abs(_rt.reshape(-1, 2) - _corner).max())
print(f"[lens] camera={CAMERA_PROFILE}  method={LENS_METHOD}  " + "  ".join(f"{k}: mean {v['mean_cm']}cm edge {v['edge_cm']}cm max {v['max_cm']}cm" for k, v in LENS_REPORT.items() if isinstance(v, dict)))

# ---------------------------------------------------------------- 2. digital-topology helpers
def f2c(p):  # floor metres -> (cx, cy), clamped (Lean: clamp_in)
    return (min(GW - 1, max(0, int(p[0] / CELL))), min(GH - 1, max(0, int(p[1] / CELL))))
def c2f(c): return np.array([(c[0] + 0.5) * CELL, (c[1] + 0.5) * CELL])

def line8(a, b):   # thin 8-connected Bresenham: used to RASTERISE barriers
    (x0, y0), (x1, y1) = a, b
    dx, dy = abs(x1 - x0), -abs(y1 - y0); sx = 1 if x0 < x1 else -1; sy = 1 if y0 < y1 else -1
    err, out = dx + dy, []
    while True:
        out.append((x0, y0))
        if (x0, y0) == (x1, y1): return out
        e2 = 2 * err
        if e2 >= dy: err += dy; x0 += sx
        if e2 <= dx: err += dx; y0 += sy

def line4(a, b):   # 4-connected ray: used to TEST occlusion. A 4-path cannot slip between the
    (x, y), (x1, y1) = a, b          # diagonal cells of an 8-connected barrier (4/8 duality).
    dx, dy = abs(x1 - x), abs(y1 - y); sx = 1 if x1 > x else -1; sy = 1 if y1 > y else -1
    out, ix, iy = [(x, y)], 0, 0
    for _ in range(dx + dy):
        if (0.5 + ix) * dy < (0.5 + iy) * dx: x += sx; ix += 1
        else: y += sy; iy += 1
        out.append((x, y))
    return out

STATIC = np.zeros((GH, GW), np.uint8)
for cy in range(GH):
    for cx in range(GW):
        p = c2f((cx, cy))
        if RED_RECT[0] <= p[0] <= RED_RECT[2] and RED_RECT[1] <= p[1] <= RED_RECT[3]: STATIC[cy, cx] = TOK['RED_ZONE']
        if np.hypot(p[0] - SPILL_C[0], p[1] - SPILL_C[1]) <= 1.0: STATIC[cy, cx] = TOK['HAZARD']
STATIC_BARRIER = np.zeros((GH, GW), bool)
for a, b in GUARDRAIL:
    for (cx, cy) in line8(f2c(a), f2c(b)): STATIC_BARRIER[cy, cx] = True
SEED_CELL = f2c(SPILL_C)
_rc0, _rc1 = f2c((SPILL_C[0] - RING_HALF, SPILL_C[1] - RING_HALF)), f2c((SPILL_C[0] + RING_HALF, SPILL_C[1] + RING_HALF))
FILL_BOX = (_rc0[0] - 3, _rc0[1] - 3, _rc1[0] + 3, _rc1[1] + 3)   # fill escaping this box == open perimeter

def ring_posts():
    x0, y0, L = SPILL_C[0] - RING_HALF, SPILL_C[1] - RING_HALF, 2 * RING_HALF
    pts = []
    for i in range(4): pts += [(x0 + i, y0), (x0 + L, y0 + i), (x0 + L - i, y0 + L), (x0, y0 + L - i)]
    return np.array(pts)
POSTS = ring_posts()

def seg_intersect(p, q, a, b):
    def orient(u, v, w): return (v[0] - u[0]) * (w[1] - u[1]) - (v[1] - u[1]) * (w[0] - u[0])
    d1, d2, d3, d4 = orient(a, b, p), orient(a, b, q), orient(p, q, a), orient(p, q, b)
    return (d1 * d2 < 0) and (d3 * d4 < 0)

# ---------------------------------------------------------------- 3. simulator (YOLOX-like output)
def _in_ring_box(p, m=0.6):
    return abs(p[0] - SPILL_C[0]) <= RING_HALF + m and abs(p[1] - SPILL_C[1]) <= RING_HALF + m

def _box(base_xy, height_m, wfac):
    fu = project([base_xy[0], base_xy[1], 0.0])[0]; hu = project([base_xy[0], base_xy[1], height_m])[0]
    h = fu[1] - hu[1]; w = wfac * h
    return np.array([fu[0] - w / 2, hu[1], fu[0] + w / 2, fu[1]]), h

def sim_episode(ep_seed, T=300, n_workers=6, breach=None, steam=None, bump=None):
    """bump: None | dict(t, tilt, pan) camera knocked at frame t | dict(t, blind=True) lens covered from frame t."""
    set_pose(0.0, 0.0)
    try:
        yield from _sim_episode(ep_seed, T, n_workers, breach, steam, bump)
    finally:
        set_pose(0.0, 0.0)

def _sim_episode(ep_seed, T, n_workers, breach, steam, bump):
    rng = np.random.default_rng(ep_seed)
    breach = rng.random() < 0.5 if breach is None else breach
    steam = rng.random() < 0.5 if steam is None else steam
    t_breach = int(rng.integers(80, 200)) if breach else None
    removed = int(rng.integers(0, len(POSTS))) if breach else None
    steam_win = (int(rng.integers(30, 220)), 60) if steam else None
    def new_goal():
        while True:
            r = rng.random()
            if r < 0.15: g = np.array([rng.uniform(1, 19), rng.uniform(*LANE_Y)])          # task in forklift aisle
            elif r < 0.35: g = np.array([rng.uniform(1, 19), rng.uniform(6.6, 7.6)])      # walkway hugging the rail
            else: g = np.array([rng.uniform(0.5, 19.5), rng.uniform(6.6, 14.5)])
            if not _in_ring_box(g): return g
    W = []
    for i in range(n_workers):
        p = new_goal()
        W.append(dict(id=i, pos=p, goal=new_goal(), wp=None, v=rng.uniform(0.9, 1.4),
                      helmet=rng.random() < 0.8, vest=rng.random() < 0.85,
                      t_off=int(rng.integers(50, 250)) if rng.random() < 0.3 else None, occl=0, occl_kind=None))
    fk = dict(pos=np.array([rng.uniform(2, 18), rng.uniform(3.0, 4.0)]), dir=1 if rng.random() < .5 else -1, pause=0)
    blind = False
    for t in range(T):
        if bump is not None and t == bump['t']:
            if bump.get('blind'): blind = True
            else: set_pose(bump.get('tilt', 0.0), bump.get('pan', 0.0))
        steam_on = steam_win is not None and steam_win[0] <= t < steam_win[0] + steam_win[1]
        # --- dynamics
        if fk['pause'] > 0: fk['pause'] -= 1
        else:
            fk['pos'][0] += fk['dir'] * 1.8 / FPS
            if fk['pos'][0] > 19 or fk['pos'][0] < 1: fk['dir'] *= -1
            if rng.random() < 0.01: fk['pause'] = int(rng.integers(10, 40))
        for w in W:
            if w['t_off'] is not None and t == w['t_off']: w['helmet'] = not w['helmet']
            tgt = w['wp'] if w['wp'] is not None else w['goal']
            side_now, side_goal = w['pos'][1] > 6.25, w['goal'][1] > 6.25
            if w['wp'] is None and side_now != side_goal and abs(w['pos'][0] - CROSSING[0]) > 0.8:
                w['wp'] = np.array(CROSSING); tgt = w['wp']
            d = tgt - w['pos']; n = np.linalg.norm(d)
            if n < 0.2:
                if w['wp'] is not None: w['wp'] = None
                else: w['goal'] = new_goal()
            else:
                nxt = w['pos'] + d / n * min(n, w['v'] / FPS)
                if _in_ring_box(nxt, 0.3): w['goal'] = new_goal(); w['wp'] = None
                else: w['pos'] = nxt
            if w['occl'] > 0: w['occl'] -= 1
            elif rng.random() < 0.02:
                w['occl'] = int(rng.integers(5, 20)); w['occl_kind'] = 'bottom' if rng.random() < .5 else 'top'
        # --- ground truth (continuous geometry, independent of the grid)
        gt = dict(workers={}, breach=bool(breach and t >= t_breach), blind=blind, bumped=bool(bump is not None and t >= bump['t']),
                  removed_post=POSTS[removed].tolist() if breach else None, fk=fk['pos'].copy())
        for w in W:
            dist = np.linalg.norm(w['pos'] - fk['pos'])
            blocked = any(seg_intersect(w['pos'], fk['pos'], a, b) for a, b in GUARDRAIL)
            inred = RED_RECT[0] <= w['pos'][0] <= RED_RECT[2] and RED_RECT[1] <= w['pos'][1] <= RED_RECT[3]
            gt['workers'][w['id']] = dict(pos=w['pos'].copy(), lof=bool(dist <= R_LOF and not blocked),
                                          red=bool(inred), helmet=w['helmet'], vest=w['vest'], occl=w['occl'] > 0)
        # --- detections
        dets = []
        for w in W:
            box, h = _box(w['pos'], 1.75, 0.42)
            size_f = float(np.clip(h / 28.0, 0.6, 1.0))
            occl = w['occl'] > 0
            p_det = (0.35 if occl else 0.97) * (0.85 if steam_on else 1.0) * size_f * (0.03 if blind else 1.0)
            if rng.random() < p_det:
                b = box + rng.normal(0, 0.03 * h, 4)
                head_vis = torso_vis = True
                if occl and w['occl_kind'] == 'bottom': b[3] = b[1] + 0.55 * h
                if occl and w['occl_kind'] == 'top': b[1] = b[1] + 0.35 * h; head_vis = False; torso_vis = rng.random() < .5
                conf = rng.beta(3, 4) if occl else (rng.beta(4, 3) if steam_on else rng.beta(8, 2))
                dets.append(dict(cls='person', conf=float(conf), box=b.tolist(), _gt=w['id']))
                hb = np.array([box[0] + 0.4 * (box[2] - box[0]) - 0.1 * h, box[1], box[0] + 0.6 * (box[2] - box[0]) + 0.1 * h, box[1] + 0.15 * h])
                if head_vis:
                    if w['helmet'] and rng.random() < 0.9 * size_f:
                        dets.append(dict(cls='helmet', conf=float(rng.beta(7, 2) * (0.75 if steam_on else 1)), box=(hb + rng.normal(0, .02 * h, 4)).tolist(), _gt=w['id']))
                    elif (not w['helmet']) and rng.random() < 0.15:
                        dets.append(dict(cls='helmet', conf=float(rng.beta(2, 4)), box=(hb + rng.normal(0, .02 * h, 4)).tolist(), _gt=w['id']))
                vb = np.array([box[0], box[1] + 0.22 * h, box[2], box[1] + 0.55 * h])
                if torso_vis:
                    if w['vest'] and rng.random() < 0.92 * size_f:
                        dets.append(dict(cls='vest', conf=float(rng.beta(7, 2) * (0.75 if steam_on else 1)), box=(vb + rng.normal(0, .02 * h, 4)).tolist(), _gt=w['id']))
                    elif (not w['vest']) and rng.random() < 0.10:
                        dets.append(dict(cls='vest', conf=float(rng.beta(2, 5)), box=(vb + rng.normal(0, .02 * h, 4)).tolist(), _gt=w['id']))
        if rng.random() < 0.05:   # background FP person
            p = rng.uniform([0, 0], [20, 15]); box, h = _box(p, 1.75, 0.42)
            dets.append(dict(cls='person', conf=float(rng.beta(2, 5)), box=box.tolist(), _gt=None))
        if rng.random() < (0.03 if blind else 0.95):
            box, h = _box(fk['pos'], 2.2, 1.1)
            dets.append(dict(cls='forklift', conf=float(rng.beta(8, 2)), box=(box + rng.normal(0, .03 * h, 4)).tolist(), _gt='fk'))
        for k, pp in enumerate(POSTS):
            if breach and t >= t_breach and k == removed: continue
            if rng.random() < (0.45 if steam_on else 0.8) * (0.03 if blind else 1.0):
                box, h = _box(pp, 1.0, 0.25)
                dets.append(dict(cls='barrier_post', conf=float(rng.beta(6, 2)), box=(box + rng.normal(0, .03 * h, 4)).tolist(), _gt=('post', k)))
        yield t, dets, gt

# ---------------------------------------------------------------- 3b. grey frames + camera-drift monitor
# The simulator renders a grey frame of the textured floor through the TRUE (possibly bumped) camera and lens.
# The monitor matches ORB features against a reference frame grabbed at calibration time. For a planar floor any
# camera motion is a homography G between undistorted views, so it can be corrected exactly: floor = H_ref.G.und(uv).
TEX_PPM, TEX_X0, TEX_Y0, TEX_WM, TEX_HM = 20, -20.0, -10.0, 60.0, 70.0
def _make_texture(seed=11):
    r = np.random.default_rng(seed); W, H = int(TEX_WM * TEX_PPM), int(TEX_HM * TEX_PPM)
    tex = np.full((H, W), 118, np.uint8)
    for _ in range(4000):
        c = (int(r.integers(0, W)), int(r.integers(0, H))); g = int(r.integers(30, 230))
        if r.random() < .5: cv2.circle(tex, c, int(r.integers(3, 18)), g, -1)
        else: cv2.rectangle(tex, c, (c[0] + int(r.integers(5, 40)), c[1] + int(r.integers(5, 40))), g, -1)
    for k in range(0, W, TEX_PPM): cv2.line(tex, (k, 0), (k, H), 70, 1)
    for k in range(0, H, TEX_PPM): cv2.line(tex, (0, k), (W, k), 70, 1)
    return cv2.GaussianBlur(tex, (3, 3), 0)
TEX = _make_texture(); _MAPS = {}
def _render_maps():
    key = tuple(np.round(CAM_POSE['R'], 9).ravel())
    if key not in _MAPS:
        u, v = np.meshgrid(np.arange(IMG_W, dtype=np.float64), np.arange(IMG_H, dtype=np.float64))
        n = und_pts(np.c_[u.ravel(), v.ravel()], K_CAM, DIST_TRUE)            # TRUE lens: this is the physics
        d = np.c_[(n[:, 0] - K_CAM[0, 2]) / K_CAM[0, 0], (n[:, 1] - K_CAM[1, 2]) / K_CAM[1, 1], np.ones(len(n))] @ CAM_POSE['R']
        with np.errstate(divide='ignore', invalid='ignore'):
            t = -CAM_C[2] / d[:, 2]
        X, Y = CAM_C[0] + t * d[:, 0], CAM_C[1] + t * d[:, 1]
        mx, my = ((X - TEX_X0) * TEX_PPM).astype(np.float32), ((Y - TEX_Y0) * TEX_PPM).astype(np.float32)
        bad = (d[:, 2] >= 0) | ~np.isfinite(mx) | ~np.isfinite(my)
        mx[bad] = -1; my[bad] = -1
        _MAPS[key] = (mx.reshape(IMG_H, IMG_W), my.reshape(IMG_H, IMG_W), bad.reshape(IMG_H, IMG_W))
    return _MAPS[key]
def render_frame(rng, dets=(), blind=False):
    mx, my, bad = _render_maps()
    img = cv2.remap(TEX, mx, my, cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0).astype(np.float32)
    img[bad] = rng.normal(150, 25, int(bad.sum()))                          # sky/walls: no stable features
    for d in dets:                                                           # moving clutter: people, forklifts
        if d['cls'] in ('person', 'forklift'):
            x1, y1, x2, y2 = map(int, d['box']); img[max(0, y1):max(0, y2), max(0, x1):max(0, x2)] = 55
    if blind: img = img * 0.06 + 8
    return np.clip(img + rng.normal(0, 4, img.shape), 0, 255).astype(np.uint8)

class DriftMonitor:
    """ORB + RANSAC homography between the current grey frame and the calibration-time reference, in undistorted
    pixels. Returns {valid, G (current->reference), drift_px, inliers, ratio}. Not enough consistent matches
    (lens covered, steam, camera turned away) => valid=False => rules go UNKNOWN."""
    def __init__(self, ref_frame, calib):
        self.K, self.D = calib['K'], calib['D']; self.orb = cv2.ORB_create(2000, fastThreshold=12)
        kp, self.ref_des = self.orb.detectAndCompute(ref_frame, None)
        self.ref_pts = und_pts([k.pt for k in kp], self.K, self.D)
        self.bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    def check(self, frame):
        bad = dict(valid=False, G=None, drift_px=None, inliers=0, ratio=0.0)
        kp, des = self.orb.detectAndCompute(frame, None)
        if des is None or len(kp) < 30: return bad
        m = self.bf.match(des, self.ref_des)
        if len(m) < DRIFT_MIN_INLIERS: return dict(bad, inliers=len(m))
        cur = und_pts([kp[x.queryIdx].pt for x in m], self.K, self.D); ref = self.ref_pts[[x.trainIdx for x in m]]
        G, mask = cv2.findHomography(cur, ref, cv2.RANSAC, 2.0)
        if G is None: return bad
        inl = int(mask.sum()); ratio = inl / len(m)
        drift = float(np.median(np.linalg.norm(apply_h(np.linalg.inv(G), self.ref_pts) - self.ref_pts, axis=1)))
        return dict(valid=bool(inl >= DRIFT_MIN_INLIERS and ratio >= DRIFT_MIN_RATIO), G=G.tolist(),
                    drift_px=round(drift, 3), inliers=inl, ratio=round(ratio, 3))

def yolox_to_dets(outputs, class_names, ratio=1.0, conf_thr=0.1):
    """Adapter for real YOLOX postprocess() output: tensor [N,7] = x1,y1,x2,y2,obj,cls_conf,cls_id."""
    dets = []
    if outputs is None: return dets
    for row in np.asarray(outputs):
        conf = float(row[4] * row[5])
        if conf >= conf_thr: dets.append(dict(cls=class_names[int(row[6])], conf=conf, box=(row[:4] / ratio).tolist()))
    return dets

# ---------------------------------------------------------------- 4. hysteresis + 3-valued PPE (mirrors Lean defs)
def step_s(s, pos): return (s + 1 if s < S_MAX else S_MAX) if pos else max(0, s - 1)   # Lean: stepS
def alarm_next(alarm, s): return True if s >= HI else (False if s <= LO else alarm)
OK, VIOL, UNK = 'compliant', 'violation', 'unknown'
def decide_ppe(occluded, has_ok, has_viol):                                            # Lean: decidePPE
    if occluded: return UNK
    if has_ok and not has_viol: return OK
    if has_viol and not has_ok: return VIOL
    return UNK

# ---------------------------------------------------------------- 5. PPE fusion model (SAM-trained) + Mondrian SCCP
PPE_WIN = 10
def ppe_features(hist):
    """hist: deque of (detected, occluded, pconf, hconf, vconf, box_h). Returns (13-dim feature, occluded_now)."""
    det_now, occ_now, p_now, h_now, v_now, bh_now = hist[-1]
    good = [x for x in hist if x[0] and not x[1]]
    hc = np.array([x[3] for x in good]) if good else np.zeros(1)
    vc = np.array([x[4] for x in good]) if good else np.zeros(1)
    bh = np.mean([x[5] for x in good]) if good else 0.0
    f = [h_now, v_now, p_now, float(det_now), float(occ_now), hc.mean(), hc.max(), (hc > .5).mean(),
         vc.mean(), vc.max(), (vc > .5).mean(), len(good) / PPE_WIN, bh / 50.0]
    return np.array(f, np.float32), bool(occ_now or not det_now)

class MLP(nn.Module):
    def __init__(self, d):
        super().__init__(); self.net = nn.Sequential(nn.Linear(d, 32), nn.GELU(), nn.Linear(32, 32), nn.GELU(), nn.Linear(32, 2))
    def forward(self, x): return self.net(x)

class SAM(torch.optim.Optimizer):
    def __init__(self, params, base_cls, rho=0.05, **kw):
        super().__init__(params, dict(rho=rho, **kw))
        self.base = base_cls(self.param_groups, **kw); self.param_groups = self.base.param_groups
    @torch.no_grad()
    def first_step(self):
        gn = torch.norm(torch.stack([p.grad.norm(2) for g in self.param_groups for p in g['params'] if p.grad is not None]), 2)
        for g in self.param_groups:
            sc = g['rho'] / (gn + 1e-12)
            for p in g['params']:
                if p.grad is None: continue
                e = p.grad * sc; p.add_(e); self.state[p]['e'] = e
    @torch.no_grad()
    def second_step(self):
        for g in self.param_groups:
            for p in g['params']:
                if 'e' in self.state[p]: p.sub_(self.state[p]['e'])
        self.base.step()

class PPEHead:
    def __init__(self, model, mu, sd, q): self.model, self.mu, self.sd, self.q = model, mu, sd, q
    @torch.no_grad()
    def proba(self, X):
        X = torch.tensor((np.atleast_2d(X) - self.mu) / self.sd, dtype=torch.float32)
        return torch.softmax(self.model(X), -1).numpy()
    def decide(self, f, occluded):
        p = self.proba(f)[0]
        return decide_ppe(occluded, bool(1 - p[0] <= self.q[0]), bool(1 - p[1] <= self.q[1])), p

def mondrian_fit(p_cal, y_cal, alpha=ALPHA):
    q = {}
    for c in (0, 1):
        s = np.sort(1 - p_cal[y_cal == c, c]); n = len(s)
        assert n >= N_CAL_FORMAL_BOUND, f"class {c}: n_cal={n} < {N_CAL_FORMAL_BOUND}"
        k = math.ceil((n + 1) * (1 - alpha)); q[c] = float(s[k - 1]) if k <= n else 1.0
    return q

# ---------------------------------------------------------------- 6. grammar engine
class Track:
    _n = 0
    def __init__(self, pos, f):
        Track._n += 1; self.id = Track._n; self.pos = np.array(pos, float); self.vel = np.zeros(2)
        self.last = f; self.hits = 1; self.votes = collections.Counter()
        self.s = collections.defaultdict(int); self.alarm = collections.defaultdict(bool)
        self.raw = {}; self.hist = collections.deque(maxlen=PPE_WIN); self.ppe = {'helmet': UNK, 'vest': UNK}

DRIFT_EVERY, DRIFT_OK_PX, DRIFT_MIN_INLIERS, DRIFT_MIN_RATIO = 10, 3.0, 60, 0.35
CAL_OK, CAL_CORRECTED, CAL_UNKNOWN = 'ok', 'corrected', 'unknown'

def next_calib_state(est, G_prev):
    """Drift gate (Lean: gate / lost_never_clear). est = DriftMonitor.check() output."""
    if not est['valid']: return CAL_UNKNOWN, G_prev
    if est['drift_px'] <= DRIFT_OK_PX: return CAL_OK, np.eye(3)
    return CAL_CORRECTED, np.asarray(est['G'], float)

class SafetyGrammarEngine:
    def __init__(self, ppe_heads=None, calib=None):
        self.calib = dict(calib or CALIB); self.G = np.eye(3); self.calib_state = CAL_OK; self.last_est = None
        self.tracks, self.f = [], -1
        self.perim_s, self.perim_alarm = 0, False; self.ppe_heads = ppe_heads or {}; self.posts = []

    # R4 part-whole: PERSON box decomposes vertically into HEAD [0,.25] . TORSO [.2,.6] . LEGS
    @staticmethod
    def _parts(pbox, parts):
        x1, y1, x2, y2 = pbox; h = y2 - y1; out = {}
        for cls, lo, hi in (('helmet', -0.05, 0.25), ('vest', 0.15, 0.62)):
            best = 0.0
            for d in parts.get(cls, []):
                cx, cy = (d['box'][0] + d['box'][2]) / 2, (d['box'][1] + d['box'][3]) / 2
                if x1 <= cx <= x2 and y1 + lo * h <= cy <= y1 + hi * h: best = max(best, d['conf'])
            out[cls] = best
        return out

    def _update_posts(self, pts):
        """Incremental POST terminals: each cluster keeps a running mean and the frames it was seen in.
        Persistent iff seen in >= POST_MIN of the last POST_WIN frames. O(points x clusters) per frame."""
        f = self.f
        for p in pts:
            if self.posts:
                M = np.array([c['xy'] for c in self.posts]); d = np.linalg.norm(M - p, axis=1); k = int(np.argmin(d))
                if d[k] <= POST_R:
                    c = self.posts[k]; c['n'] += 1; c['xy'] += (p - c['xy']) / min(c['n'], 20); c['seen'].append(f); continue
            self.posts.append(dict(xy=p.copy(), n=1, seen=collections.deque([f])))
        for c in self.posts:
            while c['seen'] and c['seen'][0] <= f - POST_WIN: c['seen'].popleft()
        self.posts = [c for c in self.posts if c['seen']]
        return np.array([c['xy'] for c in self.posts if len(set(c['seen'])) >= POST_MIN]).reshape(-1, 2)

    @staticmethod
    def _barrier_mask(posts_xy):
        # chain grammar  BARRIER -> POST (LINK POST)*  with LINK iff |p_i - p_j| <= LINK_M, rasterised 8-connected
        bar = STATIC_BARRIER.copy(); pts = np.asarray(posts_xy).reshape(-1, 2)
        for i in range(len(pts)):
            for j in range(i + 1, len(pts)):
                if np.linalg.norm(pts[i] - pts[j]) <= LINK_M:
                    for (cx, cy) in line8(f2c(pts[i]), f2c(pts[j])): bar[cy, cx] = True
        return bar

    @staticmethod
    def _flood_open(bar):
        x0, y0, x1, y1 = FILL_BOX; seen = {SEED_CELL}; q = collections.deque([SEED_CELL])
        while q:
            cx, cy = q.popleft()
            if cx <= x0 or cy <= y0 or cx >= x1 or cy >= y1: return True
            for nx, ny in ((cx + 1, cy), (cx - 1, cy), (cx, cy + 1), (cx, cy - 1)):   # 4-connected fill
                if (nx, ny) not in seen and not bar[ny, nx]: seen.add((nx, ny)); q.append((nx, ny))
        return False

    def _closure(self, posts_xy):
        bar = self._barrier_mask(posts_xy); return self._flood_open(bar), bar

    def _rules(self, ctx):
        """Native Python rule back end. ctx = {bar, fk_cells, workers:{tid: cell}}.
        Returns (perimeter_open, {tid: {'lof','red'}}, proofs). Overridden by the yappu Datalog back end."""
        bar, raw = ctx['bar'], {}
        for tid, wc in ctx['workers'].items():
            lof = False
            for fc in ctx['fk_cells']:   # R1: within radius AND 4-connected ray hits no BARRIER token
                if np.hypot(*(c2f(wc) - c2f(fc))) <= R_LOF and not any(bar[cy, cx] for cx, cy in line4(wc, fc)[1:-1]): lof = True
            raw[tid] = dict(lof=lof, red=bool(STATIC[wc[1], wc[0]] == TOK['RED_ZONE']))   # R2
        return self._flood_open(bar), raw, {}

    @staticmethod
    def _gap(posts_xy):
        if len(posts_xy) < 3: return None
        P = np.asarray(posts_xy); ang = np.arctan2(P[:, 1] - SPILL_C[1], P[:, 0] - SPILL_C[0])
        P = P[np.argsort(ang)]; d = np.linalg.norm(P - np.roll(P, -1, 0), axis=1); k = int(np.argmax(d))
        return ((P[k] + P[(k + 1) % len(P)]) / 2).tolist() if d[k] > LINK_M else None

    def _to_floor(self, uv):
        # current pixels -> undistort -> (drift correction G: current -> reference view) -> calibrated homography
        return apply_h(self.calib['H'] @ self.G, und_pts(uv, self.calib['K'], self.calib['D']))

    def process(self, dets, drift_est=None):
        self.f += 1; f = self.f
        if drift_est is not None:
            self.calib_state, self.G = next_calib_state(drift_est, self.G); self.last_est = drift_est
        gated = self.calib_state == CAL_UNKNOWN
        by = collections.defaultdict(list)
        for d in dets: by[d['cls']].append(d)
        # --- persons -> floor (bottom-centre via homography); flag truncated boxes, never trust their foot point
        P = []
        for d in by['person']:
            x1, y1, x2, y2 = d['box']; h, w = y2 - y1, max(1e-3, x2 - x1)
            trunc = (h / w) < 1.9
            P.append(dict(d=d, xy=self._to_floor([[(x1 + x2) / 2, y2]])[0], trunc=trunc, h=h, parts=self._parts(d['box'], by)))
        # --- ByteTrack-style two-stage association on the FLOOR plane (metres, not pixels)
        for tr in self.tracks: tr.pred = tr.pos + tr.vel
        free_t, matched = set(range(len(self.tracks))), {}
        for stage in ('high', 'low'):
            cand = [i for i, p in enumerate(P) if i not in matched.values() and ((p['d']['conf'] >= 0.45) == (stage == 'high')) and p['d']['conf'] >= 0.15]
            pairs = sorted((np.linalg.norm(self.tracks[t].pred - P[i]['xy']), t, i) for t in free_t for i in cand)
            used = set()
            for dist, t, i in pairs:
                gate = 2.0 if P[i]['trunc'] else 1.0
                if t in free_t and i not in used and dist <= gate: free_t.discard(t); used.add(i); matched[t] = i
        for t, i in matched.items():
            tr, p = self.tracks[t], P[i]
            if not p['trunc']: tr.vel = 0.6 * tr.vel + 0.4 * (p['xy'] - tr.pos); tr.pos = p['xy'].copy()
            else: tr.pos = tr.pred
            tr.last, tr.hits = f, tr.hits + 1
            if p['d'].get('_gt') is not None: tr.votes[p['d']['_gt']] += 1
            tr.hist.append((True, p['trunc'] or p['d']['conf'] < 0.35, p['d']['conf'], p['parts']['helmet'], p['parts']['vest'], p['h']))
        for t in free_t:
            tr = self.tracks[t]; tr.pos = tr.pred; tr.vel *= 0.8; tr.hist.append((False, True, 0.0, 0.0, 0.0, 0.0))
        for i, p in enumerate(P):
            if i not in matched.values() and p['d']['conf'] >= 0.45 and not p['trunc']:
                tr = Track(p['xy'], f); tr.hist.append((True, False, p['d']['conf'], p['parts']['helmet'], p['parts']['vest'], p['h']))
                if p['d'].get('_gt') is not None: tr.votes[p['d']['_gt']] += 1
                self.tracks.append(tr)
        self.tracks = [t for t in self.tracks if f - t.last <= 15]
        # --- forklifts + barricade posts -> grid
        fks = [self._to_floor([[(d['box'][0] + d['box'][2]) / 2, d['box'][3]]])[0] for d in by['forklift'] if d['conf'] >= 0.3]
        pb = [[(d['box'][0] + d['box'][2]) / 2, d['box'][3]] for d in by['barrier_post'] if d['conf'] >= 0.3]
        cur_posts = self._to_floor(pb) if pb else np.zeros((0, 2))
        warm = self.f < POST_WIN - 1                          # verdict is UNKNOWN until the window is full
        pers_posts = self._update_posts(cur_posts)
        bar = self._barrier_mask(pers_posts)
        fk_cells = [f2c(x) for x in fks]
        conf = [tr for tr in self.tracks if tr.hits >= 3]
        self.ctx = ctx = dict(bar=bar, fk_cells=fk_cells, workers={tr.id: f2c(tr.pos) for tr in conf})
        if gated: opened, raw, proofs = None, {tid: {} for tid in ctx['workers']}, {}   # calibration lost: no verdicts
        else: opened, raw, proofs = self._rules(ctx)
        opened_1f, _ = self._closure(cur_posts)          # single-frame baseline (for evaluation only)
        if not warm and not gated: self.perim_s = step_s(self.perim_s, opened); self.perim_alarm = alarm_next(self.perim_alarm, self.perim_s)
        grid = STATIC.copy(); grid[bar] = TOK['BARRIER']
        for c in fk_cells: grid[c[1], c[0]] = TOK['FORKLIFT']
        # --- per-track hysteresis + PPE
        out = []
        for tr in conf:
            wc = ctx['workers'][tr.id]
            tr.raw = raw[tr.id]                             # {} while gated: buckets frozen, nothing decided
            for r, v in tr.raw.items(): tr.s[r] = step_s(tr.s[r], v); tr.alarm[r] = alarm_next(tr.alarm[r], tr.s[r])
            feat, occ = ppe_features(tr.hist)               # R4
            for item, head in self.ppe_heads.items(): tr.ppe[item] = head.decide(feat, occ)[0]
            grid[wc[1], wc[0]] = TOK['WORKER']
            w = dict(track=tr.id, floor_xy=tr.pos.round(2), cell=wc,
                     line_of_fire=UNK if gated else tr.alarm['lof'], red_zone=UNK if gated else tr.alarm['red'],
                     helmet=tr.ppe['helmet'], vest=tr.ppe['vest'])
            if tr.id in proofs.get('workers', {}): w['proof'] = proofs['workers'][tr.id]
            out.append(w)
        self.grid = grid
        cal = dict(state=self.calib_state, **({k: self.last_est[k] for k in ('drift_px', 'inliers', 'ratio')} if self.last_est else {}))
        return dict(frame=f, workers=out, calibration=cal, system_alerts=['calibration-lost'] if gated else [],
                    perimeter=dict(breach=None if (warm or gated) else self.perim_alarm, warming_up=warm, raw_open=opened, raw_open_single_frame=opened_1f,
                    gap_floor_xy=self._gap(pers_posts) if opened else None, posts_persistent=len(pers_posts),
                    **({'proof': proofs['perimeter']} if 'perimeter' in proofs else {})))

# ---------------------------------------------------------------- 7. metrics
def mcc(y, p):
    y, p = np.asarray(y, bool), np.asarray(p, bool)
    tp, tn, fp, fn = [float(np.sum(a)) for a in (y & p, ~y & ~p, ~y & p, y & ~p)]
    den = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return (tp * tn - fp * fn) / den if den > 0 else 0.0
def toggles(seq): s = np.asarray(seq, int); return int(np.abs(np.diff(s)).sum()) if len(s) > 1 else 0

# ---------------------------------------------------------------- 8. run the site: collect rule outputs + PPE dataset
N_EP, T_EP = 36, 300
SPLIT = dict(train=range(0, 18), cal=range(18, 24), test=range(24, 36))
LOF, RED, PER, PPE = [], [], [], []
for ep in range(N_EP):
    eng = SafetyGrammarEngine(); Track._n = 0
    seq = collections.defaultdict(lambda: collections.defaultdict(list))
    for t, dets, gt in sim_episode(1000 + ep, T=T_EP):
        rep = eng.process(dets)
        fk_box = [d['box'] for d in dets if d['cls'] == 'forklift']
        if not rep['perimeter']['warming_up']: PER.append(dict(ep=ep, gt=gt['breach'], hyst=rep['perimeter']['breach'], raw=rep['perimeter']['raw_open'],
                        single=rep['perimeter']['raw_open_single_frame'], gap=rep['perimeter']['gap_floor_xy'], removed=gt['removed_post']))
        owner = {}
        for tr in eng.tracks:
            if tr.votes and t - tr.last <= 15:
                wid, _ = tr.votes.most_common(1)[0]
                if isinstance(wid, int) and (wid not in owner or tr.last > owner[wid].last): owner[wid] = tr
        for wid, g in gt['workers'].items():
            tr = owner.get(wid)
            confirmed = tr is not None and tr.hits >= 3 and tr.raw
            pbox = [d['box'] for d in dets if d['cls'] == 'person' and d.get('_gt') == wid]
            pix = min((np.hypot((b[0] + b[2]) / 2 - (fb[0] + fb[2]) / 2, b[3] - fb[3]) for b in pbox for fb in fk_box), default=np.inf)
            LOF.append(dict(ep=ep, w=wid, t=t, gt=g['lof'], hyst=bool(confirmed and tr.alarm['lof']), raw=bool(confirmed and tr.raw['lof']), pix=pix))
            RED.append(dict(ep=ep, w=wid, gt=g['red'], hyst=bool(confirmed and tr.alarm['red'])))
            if confirmed and len(tr.hist) == PPE_WIN and t % 2 == 0:
                feat, occ = ppe_features(tr.hist)
                PPE.append(dict(ep=ep, x=feat, occ=occ, y_h=int(not g['helmet']), y_v=int(not g['vest'])))
print(f"[sim] {N_EP} episodes x {T_EP} frames processed in {time.time()-T0:.1f}s "
      f"({1000*(time.time()-T0)/(N_EP*T_EP):.2f} ms/frame incl. simulation)")

def _sel(rows, split): return [r for r in rows if r['ep'] in SPLIT[split]]
results = {}
# R1 line-of-fire: grammar vs tuned pixel-distance baseline (baseline gets ORACLE worker association)
trval = [r for r in LOF if r['ep'] < 24]; te = _sel(LOF, 'test')
taus = np.linspace(10, 200, 96)
tau = float(taus[int(np.argmax([mcc([r['gt'] for r in trval], [r['pix'] <= tt for r in trval]) for tt in taus]))])
y = [r['gt'] for r in te]
results['R1_line_of_fire'] = dict(
    test_rows=len(te), positives=int(sum(y)),
    MCC_pixel_distance_baseline=mcc(y, [r['pix'] <= tau for r in te]), tau_px=tau,
    MCC_grammar_raw=mcc(y, [r['raw'] for r in te]), MCC_grammar_hysteresis=mcc(y, [r['hyst'] for r in te]))
def _tog(key):
    g = collections.defaultdict(list)
    for r in te: g[(r['ep'], r['w'])].append(r[key])
    return sum(toggles(v) for v in g.values())
results['R1_line_of_fire'].update(alarm_toggles_gt=_tog('gt'), alarm_toggles_raw=_tog('raw'), alarm_toggles_hysteresis=_tog('hyst'))
te = _sel(RED, 'test'); results['R2_red_zone'] = dict(positives=int(sum(r['gt'] for r in te)), MCC=mcc([r['gt'] for r in te], [r['hyst'] for r in te]))
te = _sel(PER, 'test'); y = [r['gt'] for r in te]
loc = [np.linalg.norm(np.array(r['gap']) - np.array(r['removed'])) for r in te if r['gt'] and r['hyst'] and r['gap'] is not None]
delays = []
for ep in SPLIT['test']:
    rows = [r for r in PER if r['ep'] == ep]
    if rows[0]['removed'] is None: continue
    tb = next(i for i, r in enumerate(rows) if r['gt']); ta = next((i for i, r in enumerate(rows) if i >= tb and r['hyst']), None)
    if ta is not None: delays.append(ta - tb)
results['R3_perimeter'] = dict(
    breach_episodes=sum(1 for ep in SPLIT['test'] if [r for r in PER if r['ep'] == ep][0]['removed'] is not None),
    MCC_single_frame_closure=mcc(y, [r['single'] for r in te]), MCC_persistent_closure_raw=mcc(y, [r['raw'] for r in te]),
    MCC_persistent_closure_hysteresis=mcc(y, [r['hyst'] for r in te]),
    gap_localisation_error_m_median=float(np.median(loc)) if loc else None,
    detection_delay_frames_median=float(np.median(delays)) if delays else None)

# ---------------------------------------------------------------- 9. train PPE heads with SAM, calibrate with Mondrian SCCP
def train_head(item):
    tr, ca, te = _sel(PPE, 'train'), _sel(PPE, 'cal'), _sel(PPE, 'test')
    yk = 'y_h' if item == 'helmet' else 'y_v'
    Xtr = np.stack([r['x'] for r in tr]); ytr = np.array([r[yk] for r in tr])
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6
    torch.manual_seed(SEED); model = MLP(Xtr.shape[1])
    opt = SAM(model.parameters(), torch.optim.AdamW, rho=0.05, lr=3e-3, weight_decay=1e-4)
    w = torch.tensor([1.0, (ytr == 0).sum() / max(1, (ytr == 1).sum())], dtype=torch.float32)
    lossf = nn.CrossEntropyLoss(weight=w)
    X = torch.tensor((Xtr - mu) / sd, dtype=torch.float32); Y = torch.tensor(ytr)
    for epoch in range(20):
        perm = torch.randperm(len(X))
        for i in range(0, len(X), 512):
            idx = perm[i:i + 512]
            lossf(model(X[idx]), Y[idx]).backward(); opt.first_step(); opt.zero_grad()
            lossf(model(X[idx]), Y[idx]).backward(); opt.second_step(); opt.zero_grad()
    model.eval(); head = PPEHead(model, mu, sd, None)
    pc = head.proba(np.stack([r['x'] for r in ca])); yc = np.array([r[yk] for r in ca])
    head.q = mondrian_fit(pc, yc)
    Xte = np.stack([r['x'] for r in te]); yte = np.array([r[yk] for r in te]); occ = np.array([r['occ'] for r in te])
    pt = head.proba(Xte)
    base = Xte[:, 0 if item == 'helmet' else 1] < 0.5          # single-frame YOLOX threshold: no box/conf<.5 => violation
    sets = np.stack([1 - pt[:, 0] <= head.q[0], 1 - pt[:, 1] <= head.q[1]], 1)
    dec = np.array([decide_ppe(o, s[0], s[1]) for o, s in zip(occ, sets)])
    m = dec != UNK
    cov = {c: float(np.mean(sets[yte == c, c])) for c in (0, 1)}
    return head, dict(test_rows=int(len(yte)), violations=int(yte.sum()), n_cal_per_class={c: int((yc == c).sum()) for c in (0, 1)},
                      MCC_single_frame_threshold=mcc(yte, base), MCC_SAM_fusion_argmax=mcc(yte, pt.argmax(1) == 1),
                      MCC_SCCP_decided=mcc(yte[m], dec[m] == VIOL), unknown_rate=float(1 - m.mean()),
                      sccp_coverage_per_class=cov, target_coverage=1 - ALPHA,
                      false_violation_rate_when_occluded_baseline=float(np.mean(base[occ & (yte == 0)])) if np.any(occ & (yte == 0)) else None,
                      compliant_emitted_when_occluded=int(np.sum((dec == OK) & occ)))
HEADS = {}
for item in ('helmet', 'vest'):
    HEADS[item], results[f'R4_ppe_{item}'] = train_head(item)

# ---------------------------------------------------------------- 10. Lean 4 (kernel-only) proofs + Python mirror checks
LEAN_SRC = r'''
-- VG360 array-grammar safety layer: kernel-only (no Mathlib). Pinned toolchain: see lean-toolchain.
-- (1) Leaky-bucket hysteresis: score moves by <= 1 per frame, so any OFF(s<=LO) -> ON(s>=HI)
--     or ON -> OFF transition takes >= HI-LO frames (no flicker), and persistent evidence fires.
def stepS (S : Nat) : Bool → Nat → Nat
  | true,  s => if s < S then s + 1 else S
  | false, s => s - 1

def run (S : Nat) : List Bool → Nat → Nat
  | [],      s => s
  | p :: ps, s => run S ps (stepS S p s)

theorem step_bound (S s : Nat) (p : Bool) (h : s ≤ S) : stepS S p s ≤ S := by
  cases p with
  | false => show s - 1 ≤ S; omega
  | true  => show (if s < S then s + 1 else S) ≤ S; split <;> omega

theorem step_up (S s : Nat) (p : Bool) : stepS S p s ≤ s + 1 := by
  cases p with
  | false => show s - 1 ≤ s + 1; omega
  | true  => show (if s < S then s + 1 else S) ≤ s + 1; split <;> omega

theorem step_down (S s : Nat) (p : Bool) (h : s ≤ S) : s ≤ stepS S p s + 1 := by
  cases p with
  | false => show s ≤ s - 1 + 1; omega
  | true  => show s ≤ (if s < S then s + 1 else S) + 1; split <;> omega

theorem run_bound (S : Nat) : ∀ (ps : List Bool) (s : Nat), s ≤ S → run S ps s ≤ S
  | [],      _, h => h
  | p :: ps, s, h => run_bound S ps (stepS S p s) (step_bound S s p h)

theorem run_up (S : Nat) : ∀ (ps : List Bool) (s : Nat), run S ps s ≤ s + ps.length
  | [],      s => by show s ≤ s + 0; omega
  | p :: ps, s => by
      have h1 := step_up S s p
      have h2 := run_up S ps (stepS S p s)
      show run S ps (stepS S p s) ≤ s + (ps.length + 1)
      omega

theorem run_down (S : Nat) : ∀ (ps : List Bool) (s : Nat), s ≤ S → s ≤ run S ps s + ps.length
  | [],      s, _ => by show s ≤ s + 0; omega
  | p :: ps, s, h => by
      have h1 := step_down S s p h
      have h2 := run_down S ps (stepS S p s) (step_bound S s p h)
      show s ≤ run S ps (stepS S p s) + (ps.length + 1)
      omega

theorem no_fast_rise (S LO HI s : Nat) (ps : List Bool)
    (hlo : s ≤ LO) (hhi : HI ≤ run S ps s) : HI - LO ≤ ps.length := by
  have := run_up S ps s; omega

theorem no_fast_fall (S LO HI s : Nat) (ps : List Bool) (hs : s ≤ S)
    (hhi : HI ≤ s) (hlo : run S ps s ≤ LO) : HI - LO ≤ ps.length := by
  have := run_down S ps s hs; omega

theorem persistent_fires (S : Nat) : ∀ (n s : Nat), s + n ≤ S → run S (List.replicate n true) s = s + n
  | 0,     _, _ => rfl
  | n + 1, s, h => by
      have hlt : s < S := by omega
      have hstep : stepS S true s = s + 1 := by
        show (if s < S then s + 1 else S) = s + 1
        rw [if_pos hlt]
      show run S (List.replicate n true) (stepS S true s) = s + (n + 1)
      rw [hstep, persistent_fires S n (s + 1) (by omega)]
      omega

-- (2) 3-valued PPE decision: an occluded person is NEVER reported compliant; an ambiguous
--     conformal set {ok, violation} is never collapsed to a verdict.
inductive PPE where
  | compliant | violation | unknown
  deriving DecidableEq

def decidePPE : Bool → Bool → Bool → PPE
  | true,  _,     _     => .unknown
  | false, true,  false => .compliant
  | false, false, true  => .violation
  | false, _,     _     => .unknown

theorem occluded_never_compliant (a b : Bool) : decidePPE true a b ≠ PPE.compliant := by
  cases a <;> cases b <;> decide

theorem compliant_iff (o a b : Bool) :
    decidePPE o a b = PPE.compliant ↔ (o = false ∧ a = true ∧ b = false) := by
  cases o <;> cases a <;> cases b <;> decide

theorem ambiguous_is_unknown (o : Bool) : decidePPE o true true = PPE.unknown := by
  cases o <;> rfl

-- (3) Grid indexing never leaves the array; line-of-fire trigger is symmetric and monotone in radius.
def clampIdx (W x : Nat) : Nat := if x < W then x else W - 1

theorem clamp_in (W x : Nat) (hW : 0 < W) : clampIdx W x < W := by
  show (if x < W then x else W - 1) < W
  split <;> omega

def adist (a b : Nat) : Nat := (a - b) + (b - a)

theorem adist_symm (a b : Nat) : adist a b = adist b a := by
  show (a - b) + (b - a) = (b - a) + (a - b); omega

theorem lof_radius_mono (a b r r' : Nat) (h : adist a b ≤ r) (hr : r ≤ r') : adist a b ≤ r' := by
  omega

-- (4) Proof-carrying breach alerts. The Datalog derivation of a perimeter breach is a cell path;
--     the runtime re-checks it with validPath. Checker-true => every step is a 4-neighbour move and
--     the head cell is free, so the certificate cannot cross a barrier diagonally (4/8 duality).
def adj4 (a b : Nat × Nat) : Bool :=
  (a.1 == b.1 && (a.2 + 1 == b.2 || b.2 + 1 == a.2)) ||
  (a.2 == b.2 && (a.1 + 1 == b.1 || b.1 + 1 == a.1))

def validPath (blocked : Nat × Nat → Bool) : List (Nat × Nat) → Bool
  | []            => true
  | [c]           => !blocked c
  | c :: d :: rest => !blocked c && adj4 c d && validPath blocked (d :: rest)

theorem valid_step (b : Nat × Nat → Bool) (c d : Nat × Nat) (rest : List (Nat × Nat))
    (h : validPath b (c :: d :: rest) = true) :
    adj4 c d = true ∧ validPath b (d :: rest) = true := by
  have h' : (!b c && adj4 c d && validPath b (d :: rest)) = true := h
  revert h'
  cases b c <;> cases adj4 c d <;> cases validPath b (d :: rest) <;> decide

theorem valid_head_free (b : Nat × Nat → Bool) (c : Nat × Nat) (rest : List (Nat × Nat))
    (h : validPath b (c :: rest) = true) : b c = false := by
  cases rest with
  | nil =>
      have h' : (!b c) = true := h
      revert h'; cases b c <;> decide
  | cons d r =>
      have h' : (!b c && adj4 c d && validPath b (d :: r)) = true := h
      revert h'; cases b c <;> cases adj4 c d <;> cases validPath b (d :: r) <;> decide

theorem valid_all_free (b : Nat × Nat → Bool) :
    ∀ (p : List (Nat × Nat)), validPath b p = true → ∀ x, x ∈ p → b x = false
  | [], _, x, hx => nomatch hx
  | [c], h, x, hx => by
      cases hx with
      | head => exact valid_head_free b c [] h
      | tail _ h2 => exact nomatch h2
  | c :: d :: rest, h, x, hx => by
      cases hx with
      | head => exact valid_head_free b c (d :: rest) h
      | tail _ h2 => exact valid_all_free b (d :: rest) (valid_step b c d rest h).2 x h2

-- (5) Calibration gate. When the drift monitor cannot confirm the camera still sees the calibrated view,
--     no rule may say "clear" (or "alarm"): every geometric verdict is UNKNOWN until calibration is trusted.
inductive Calib where
  | ok | corrected | unknown
  deriving DecidableEq

inductive Verdict where
  | alarm | clear | unknown
  deriving DecidableEq

def gate : Calib → Bool → Verdict
  | .unknown, _     => .unknown
  | _,        true  => .alarm
  | _,        false => .clear

theorem lost_never_clear (b : Bool) : gate Calib.unknown b ≠ Verdict.clear := by
  cases b <;> decide

theorem lost_never_alarm (b : Bool) : gate Calib.unknown b ≠ Verdict.alarm := by
  cases b <;> decide

theorem trusted_passes (c : Calib) (b : Bool) (h : c ≠ Calib.unknown) :
    gate c b = (if b then Verdict.alarm else Verdict.clear) := by
  cases c with
  | unknown   => exact absurd rfl h
  | ok        => cases b <;> rfl
  | corrected => cases b <;> rfl

#print axioms no_fast_rise
#print axioms lost_never_clear
#print axioms valid_all_free
#print axioms persistent_fires
#print axioms occluded_never_compliant
'''
LEAN_TOOLCHAIN = 'leanprover/lean4:v4.31.0'
os.makedirs('vg360_lean', exist_ok=True)
open('vg360_lean/VG360Grammar.lean', 'w').write(LEAN_SRC)
open('vg360_lean/lean-toolchain', 'w').write(LEAN_TOOLCHAIN + '\n')
lean = shutil.which('lean') or (os.path.expanduser('~/.elan/bin/lean') if os.path.exists(os.path.expanduser('~/.elan/bin/lean')) else None)
if lean is None and IN_COLAB:
    subprocess.run('curl -sSfL https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh -o /tmp/elan.sh && '
                   f'bash /tmp/elan.sh -y --default-toolchain {LEAN_TOOLCHAIN}', shell=True, check=False)
    lean = os.path.expanduser('~/.elan/bin/lean') if os.path.exists(os.path.expanduser('~/.elan/bin/lean')) else None
if lean:
    r = subprocess.run([lean, 'VG360Grammar.lean'], cwd='vg360_lean', capture_output=True, text=True)
    ok = r.returncode == 0 and 'sorry' not in (r.stdout + r.stderr)
    results['lean'] = dict(compiled=ok, output=(r.stdout + r.stderr)[-1500:])
else:
    results['lean'] = dict(compiled=None, note='lean not found; file written to vg360_lean/VG360Grammar.lean')

# Python mirror of the Lean theorems (the implementation must match the spec it is proved against)
mr = np.random.default_rng(1); viol = 0
for _ in range(3000):
    s, alarm, last_off = 0, False, -1          # initial state s=0 sits at frame -1
    for k, b in enumerate(mr.random(60) < mr.uniform(.2, .8)):
        s2 = step_s(s, bool(b)); assert abs(s2 - s) <= 1 and 0 <= s2 <= S_MAX; s = s2
        new = alarm_next(alarm, s)
        if s <= LO: last_off = k
        if new and not alarm and (k - last_off) < HI - LO: viol += 1
        alarm = new
assert viol == 0
for o in (0, 1):
    for a in (0, 1):
        for b in (0, 1):
            d = decide_ppe(bool(o), bool(a), bool(b))
            assert (d == OK) == (o == 0 and a == 1 and b == 0) and not (o and d == OK)
results['python_mirror_checks'] = 'passed (3000 random hysteresis traces + full PPE truth table)'  # valid_path checks run in 10b

# ---------------------------------------------------------------- 10b. yappu-fsm Datalog back end for the 2D grammar
# The grid rules are written as controlled-English sentences in yappu-fsm's style ("If ... and ... then ...
# alert is raised ..."), compiled to yappu's clause schema {rid, base, head, body, stratum, text}, and run by
#   YappuDatalog   : faithful port of yappu_fsm.rules.datalog (JS RJ.run): naive per-stratum fixpoint,
#                    stratified negation, provenance via why(); same semantics as the package
#   IndexedDatalog : same fixpoint, semi-naive + argument indexes (an engine upgrade proposal for yappu)
# Every alert carries its derivation. For a perimeter breach the derivation IS the escape path from the
# hazard to the fill boundary, so it pinpoints the gap; it is re-checked by valid_path (Lean: validPath).
import re as _re

def _cid(c): return f"{c[0]}:{c[1]}"
def _cxy(a): x, y = a.split(':'); return int(x), int(y)

GRAMMAR_RULES = [
    ('SEED',   "If a cell C is a hazard-cell then the cell C is reached."),
    ('SPREAD', "If the cell C is reached and the cell C is adjacent to a cell D and the cell D is not a barrier-cell then the cell D is reached."),
    ('BREACH', "If the cell C is reached and the cell C is a border-cell then a perimeter-breach alert is raised."),
    ('SHIELD', "If a cell C lies on the ray from a person T to a forklift K and the cell C is a barrier-cell then the person T is shielded from the forklift K."),
    ('LOF',    f"If a person T is within {R_LOF} metres of a forklift K and the person T is not shielded from the forklift K then a line-of-fire alert is raised for the person T."),
    ('REDZONE', "If a person T stands on a cell C and the cell C is a red-zone-cell then a red-zone alert is raised for the person T."),
]
# condition / conclusion phrase -> literal. Stand-in for yappu's ACE parser: these patterns (adjacency,
# recursion through "reached", "not a barrier-cell") are what its sentence grammar would need to accept.
_V = r'([A-Z])'
_PHRASES = [
    (rf'(?:a|the) cell {_V} is a hazard-cell', lambda m: ('hazard', [m[1]], False)),
    (rf'(?:a|the) cell {_V} is reached', lambda m: ('reached', [m[1]], False)),
    (rf'(?:a|the) cell {_V} is adjacent to (?:a|the) cell {_V}', lambda m: ('adjacent', [m[1], m[2]], False)),
    (rf'(?:a|the) cell {_V} is not a barrier-cell', lambda m: ('barrier', [m[1]], True)),
    (rf'(?:a|the) cell {_V} is a barrier-cell', lambda m: ('barrier', [m[1]], False)),
    (rf'(?:a|the) cell {_V} is a border-cell', lambda m: ('border', [m[1]], False)),
    (rf'(?:a|the) cell {_V} is a red-zone-cell', lambda m: ('red', [m[1]], False)),
    (rf'(?:a|the) cell {_V} lies on the ray from (?:a|the) person {_V} to (?:a|the) forklift {_V}', lambda m: ('on_ray', [m[2], m[3], m[1]], False)),
    (rf'(?:a|the) person {_V} is within ([0-9.]+) metres of (?:a|the) forklift {_V}', lambda m: ('within', [m[1], m[3], m[2]], False)),
    (rf'(?:a|the) person {_V} is not shielded from (?:a|the) forklift {_V}', lambda m: ('shielded', [m[1], m[2]], True)),
    (rf'(?:a|the) person {_V} is shielded from (?:a|the) forklift {_V}', lambda m: ('shielded', [m[1], m[2]], False)),
    (rf'(?:a|the) person {_V} stands on (?:a|the) cell {_V}', lambda m: ('stands_on', [m[1], m[2]], False)),
    (rf'an? ([a-z-]+) alert is raised for (?:a|the) person {_V}', lambda m: ('alert', [m[2], m[1]], False)),
    (rf'an? ([a-z-]+) alert is raised', lambda m: ('alert', ['scene', m[1]], False)),
]
def _lit(phrase):
    phrase = phrase.strip()
    for pat, mk in _PHRASES:
        m = _re.fullmatch(pat, phrase)
        if m:
            pred, args, neg = mk(m)
            return dict(pred=pred, neg=neg, args=[{'v': a} if _re.fullmatch(r'[A-Z]', a) else a for a in args])
    raise ValueError(f'cannot parse: {phrase!r}')

def compile_sentences(rules):
    clauses = []
    for rid, text in rules:
        m = _re.fullmatch(r'If (.+) then (.+)\.', text)
        body = [_lit(p) for p in m[1].split(' and ')]; head = _lit(m[2])
        clauses.append(dict(rid=rid, base=rid, head=head, body=body, stratum=0, text=''))
    idb = {c['head']['pred'] for c in clauses}; st = {p: 0 for p in idb}
    for _ in range(len(idb) + 1):                       # stratify: negated IDB literal => strictly higher stratum
        for c in clauses:
            h = c['head']['pred']
            for l in c['body']:
                if l['pred'] in idb: st[h] = max(st[h], st[l['pred']] + (1 if l['neg'] else 0))
    for c in clauses:
        c['stratum'] = st[c['head']['pred']]
        fmt = lambda l: ('not ' if l['neg'] else '') + f"{l['pred']}(" + ', '.join(a['v'] if isinstance(a, dict) else f'"{a}"' for a in l['args']) + ')'
        c['text'] = f"{fmt(c['head'])} :- {', '.join(fmt(l) for l in c['body'])}."
    return clauses
GRAMMAR_CLAUSES = compile_sentences(GRAMMAR_RULES)
RULE_SRC = dict(GRAMMAR_RULES)

def _ground(l, b): return (l['pred'],) + tuple(b[a['v']] if isinstance(a, dict) else a for a in l['args'])

class YappuDatalog:
    """Faithful port of yappu_fsm.rules.datalog (as RJ.run in the JS port)."""
    def __init__(self, clauses, facts):
        self.F, self.by, self.W = {}, collections.defaultdict(list), {}
        for f in facts: self._add(tuple(f), ('fact', []))
        for s in sorted({c['stratum'] for c in clauses}):
            cs = [c for c in clauses if c['stratum'] == s]; changed = True
            while changed:
                changed = False
                for c in cs:
                    for b, used in list(self._match(c['body'], 0, {}, [])):
                        if self._add(_ground(c['head'], b), (c['rid'], used)): changed = True
    def _add(self, f, w):
        if f in self.F: return False
        self.F[f] = True; self.by[f[0]].append(f); self.W[f] = w; return True
    def _match(self, body, i, b, used):
        if i == len(body): yield b, used; return
        l = body[i]
        if l['neg']:
            g = _ground(l, b)
            if g not in self.F: yield from self._match(body, i + 1, b, used + [('not',) + g])
            return
        for f in list(self.by.get(l['pred'], [])):
            if len(f) != len(l['args']) + 1: continue
            nb, ok = dict(b), True
            for j, a in enumerate(l['args']):
                v = f[j + 1]
                if isinstance(a, dict):
                    if a['v'] in nb and nb[a['v']] != v: ok = False; break
                    nb[a['v']] = v
                elif a != v: ok = False; break
            if ok: yield from self._match(body, i + 1, nb, used + [f])
    def has(self, f): return tuple(f) in self.F
    def query(self, p): return self.by.get(p, [])
    def why(self, f): return self.W.get(tuple(f), ('?', []))

class IndexedDatalog(YappuDatalog):
    """Same least fixpoint (verified fact-for-fact), semi-naive with per-argument indexes."""
    def __init__(self, clauses, facts):
        self.F, self.by, self.W, self.idx = {}, collections.defaultdict(list), {}, collections.defaultdict(list)
        self._pending = {}
        for f in facts: self._add(tuple(f), ('fact', []))
        for s in sorted({c['stratum'] for c in clauses}):
            cs = [c for c in clauses if c['stratum'] == s]; delta = None
            while True:
                new = []
                for c in cs:
                    pos = [i for i, l in enumerate(c['body']) if not l['neg']]
                    for dp in ([None] if delta is None else pos):
                        if dp is not None and not delta.get(c['body'][dp]['pred']): continue
                        for b, used in self._match_ix(c['body'], 0, {}, [], dp, delta):
                            h = _ground(c['head'], b)
                            if h not in self.F and h not in self._pending: self._pending[h] = (c['rid'], used); new.append(h)
                if not new: break
                delta = collections.defaultdict(list)
                for h in new: self._add(h, self._pending.pop(h)); delta[h[0]].append(h)
    def _add(self, f, w):
        if f in self.F: return False
        self.F[f] = True; self.by[f[0]].append(f); self.W[f] = w
        for j, v in enumerate(f[1:]): self.idx[(f[0], j, v)].append(f)
        return True
    def _match_ix(self, body, i, b, used, dp, delta):
        if i == len(body): yield b, used; return
        l = body[i]
        if l['neg']:
            g = _ground(l, b)
            if g not in self.F: yield from self._match_ix(body, i + 1, b, used + [('not',) + g], dp, delta)
            return
        if i == dp: cand = delta.get(l['pred'], [])
        else:
            cand = None
            for j, a in enumerate(l['args']):
                v = b.get(a['v']) if isinstance(a, dict) else a
                if v is not None: cand = self.idx.get((l['pred'], j, v), []); break
            if cand is None: cand = self.by.get(l['pred'], [])
        for f in cand:
            if len(f) != len(l['args']) + 1: continue
            nb, ok = dict(b), True
            for j, a in enumerate(l['args']):
                v = f[j + 1]
                if isinstance(a, dict):
                    if a['v'] in nb and nb[a['v']] != v: ok = False; break
                    nb[a['v']] = v
                elif a != v: ok = False; break
            if ok: yield from self._match_ix(body, i + 1, nb, used + [f], dp, delta)

def explain(m, f, depth=0, seen=None):
    """yappu-style proof lines: fact, the rule that derived it (with its sentence), then its premises."""
    seen = seen or set(); f = tuple(f)
    if f[0] == 'not': return [f"{'  '*depth}not {f[1]}({', '.join(map(str, f[2:]))})  [absent]"]
    rid, used = m.why(f)
    out = [f"{'  '*depth}{f[0]}({', '.join(map(str, f[1:]))})  [{'given' if rid == 'fact' else rid}]"]
    if f in seen: return out
    seen.add(f)
    for u in used:
        if u[0] not in ('adjacent', 'border', 'hazard', 'red', 'on_ray'): out += explain(m, u, depth + 1, seen)
    return out

# static facts for the fill region (adjacency is data: Datalog has no arithmetic)
_X0, _Y0, _X1, _Y1 = FILL_BOX
STATIC_FACTS = [('hazard', _cid(SEED_CELL))]
for _x in range(_X0, _X1 + 1):
    for _y in range(_Y0, _Y1 + 1):
        if _x in (_X0, _X1) or _y in (_Y0, _Y1): STATIC_FACTS.append(('border', _cid((_x, _y))))
        else:        # adjacency only out of interior cells (the BFS stops at the border)
            for _nx, _ny in ((_x + 1, _y), (_x - 1, _y), (_x, _y + 1), (_x, _y - 1)): STATIC_FACTS.append(('adjacent', _cid((_x, _y)), _cid((_nx, _ny))))
for _y in range(GH):
    for _x in range(GW):
        if STATIC[_y, _x] == TOK['RED_ZONE']: STATIC_FACTS.append(('red', _cid((_x, _y))))

def grid_facts(ctx):
    bar = ctx['bar']; F = list(STATIC_FACTS)
    F += [('barrier', _cid((x, y))) for y, x in zip(*np.nonzero(bar))]
    for tid, wc in ctx['workers'].items():
        T = f'T{tid}'; F.append(('stands_on', T, _cid(wc)))
        for k, fc in enumerate(ctx['fk_cells']):
            if np.hypot(*(c2f(wc) - c2f(fc))) <= R_LOF:          # geometry -> fact, like yappu's in_zone_for timers
                K = f'F{k}'; F.append(('within', T, K, str(R_LOF)))
                F += [('on_ray', T, K, _cid(c)) for c in line4(wc, fc)[1:-1]]
    return F

def escape_path(m):
    """Walk the provenance of alert(scene, perimeter-breach) back to the hazard: the escape path."""
    hits = [f for f in m.query('alert') if f[1] == 'scene' and f[2] == 'perimeter-breach']
    if not hits: return None
    rid, used = m.why(hits[0]); cur = next(u for u in used if u[0] == 'reached'); path = []
    while True:
        path.append(_cxy(cur[1])); rid, used = m.why(cur)
        if rid == 'SEED': break
        cur = next(u for u in used if u[0] == 'reached')
    return path[::-1]

def valid_path(path, bar):   # Python mirror of Lean validPath + endpoint checks
    if not path or path[0] != SEED_CELL: return False
    x, y = path[-1]
    if not (x in (_X0, _X1) or y in (_Y0, _Y1)): return False
    for i, (cx, cy) in enumerate(path):
        if bar[cy, cx]: return False                                 # Lean valid_all_free: every cell free
        if i + 1 < len(path):
            nx, ny = path[i + 1]
            if abs(cx - nx) + abs(cy - ny) != 1: return False
    return True

def ring_exit(path):
    """First escape-path cell outside the designed barricade square: where the chain is broken."""
    for c in path:
        p = c2f(c)
        if abs(p[0] - SPILL_C[0]) > RING_HALF or abs(p[1] - SPILL_C[1]) > RING_HALF: return p.tolist()
    return None

class DatalogSafetyEngine(SafetyGrammarEngine):
    engine_cls = IndexedDatalog
    def _rules(self, ctx):
        m = self.engine_cls(GRAMMAR_CLAUSES, grid_facts(ctx)); self.model = m
        opened = m.has(('alert', 'scene', 'perimeter-breach'))
        raw = {tid: dict(lof=m.has(('alert', f'T{tid}', 'line-of-fire')), red=m.has(('alert', f'T{tid}', 'red-zone')))
               for tid in ctx['workers']}
        proofs = {'workers': {}}
        for tid, r in raw.items():
            if r['lof'] or r['red']:
                lines = []
                for ev in ('line-of-fire', 'red-zone'):
                    if m.has(('alert', f'T{tid}', ev)): lines += explain(m, ('alert', f'T{tid}', ev))
                proofs['workers'][tid] = lines
        if opened:
            path = escape_path(m)
            proofs['perimeter'] = dict(rule=RULE_SRC['BREACH'], escape_path=path, path_valid=valid_path(path, ctx['bar']),
                                       gap_floor_xy=ring_exit(path))
        return opened, raw, proofs

# ---- (a) cross-check the Python port against yappu's own JS engine (if node is available)
RJ_JS = r'''
const RJ = (() => {
  const SEP = "\u0001"; const key = (f) => f.join(SEP);
  function run(clauses, facts) {
    const F = new Map(), byPred = {}, why = new Map();
    const add = (f, w) => { const k = key(f); if (F.has(k)) return false; F.set(k, f); (byPred[f[0]] = byPred[f[0]] || []).push(f); why.set(k, w); return true; };
    facts.forEach((f) => add(f, ["fact", []]));
    const strata = [...new Set(clauses.map((c) => c.stratum))].sort((a, b) => a - b);
    function* match(body, i, b, used) {
      if (i === body.length) { yield [b, used]; return; }
      const l = body[i];
      if (l.neg) {
        const g = [l.pred, ...l.args.map((a) => (a && a.v ? b[a.v] : a))];
        if (!F.has(key(g))) yield* match(body, i + 1, b, [...used, ["not", ...g]]);
        return;
      }
      for (const f of (byPred[l.pred] || []).slice()) {
        if (f.length !== l.args.length + 1) continue;
        const nb = { ...b }; let ok = true;
        for (let j = 0; j < l.args.length && ok; j++) {
          const a = l.args[j], v = f[j + 1];
          if (a && a.v) { if (a.v in nb && nb[a.v] !== v) ok = false; else nb[a.v] = v; }
          else if (a !== v) ok = false;
        }
        if (ok) yield* match(body, i + 1, nb, [...used, f]);
      }
    }
    for (const s of strata) {
      const cs = clauses.filter((c) => c.stratum === s); let changed = true;
      while (changed) { changed = false;
        for (const c of cs) for (const [b, used] of [...match(c.body, 0, {}, [])]) {
          const h = [c.head.pred, ...c.head.args.map((a) => (a && a.v ? b[a.v] : a))];
          if (add(h, [c.rid, used])) changed = true; } } }
    return [...F.values()];
  }
  return { run };
})();
const inp = JSON.parse(require("fs").readFileSync(process.argv[2], "utf8"));
console.log(JSON.stringify(inp.cases.map((c) => RJ.run(inp.clauses, c).map((f) => f.join("|")).sort())));
'''  # verbatim RJ.run from the yappu-fsm tutorial (tests/test_js_port.py keeps it in sync with Python)

# ---- (b) run Python rules vs yappu Datalog on every held-out frame: agreement, speed, proof-path localisation
DL = dict(frames=0, agree_perim=0, agree_lof=0, agree_red=0, n_workers=0, t_py=[], t_ix=[], t_naive=[],
          naive_fact_equal=0, naive_checked=0, path_valid=0, path_n=0, loc_err=[])
js_cases = []
for ep in SPLIT['test']:
    eng = SafetyGrammarEngine(); Track._n = 0
    for t, dets, gt in sim_episode(1000 + ep, T=T_EP):
        eng.process(dets); ctx = eng.ctx
        t0 = time.perf_counter(); op_py, raw_py, _ = SafetyGrammarEngine._rules(eng, ctx); DL['t_py'].append(time.perf_counter() - t0)
        facts = grid_facts(ctx)
        t0 = time.perf_counter(); mi = IndexedDatalog(GRAMMAR_CLAUSES, facts); DL['t_ix'].append(time.perf_counter() - t0)
        op_dl = mi.has(('alert', 'scene', 'perimeter-breach'))
        DL['frames'] += 1; DL['agree_perim'] += op_dl == op_py
        for tid, r in raw_py.items():
            DL['n_workers'] += 1
            DL['agree_lof'] += mi.has(('alert', f'T{tid}', 'line-of-fire')) == r['lof']
            DL['agree_red'] += mi.has(('alert', f'T{tid}', 'red-zone')) == r['red']
        if op_dl:
            path = escape_path(mi); DL['path_n'] += 1; DL['path_valid'] += valid_path(path, ctx['bar'])
            ex = ring_exit(path)
            if gt['breach'] and ex is not None: DL['loc_err'].append(float(np.linalg.norm(np.array(ex) - np.array(gt['removed_post']))))
        if t % 25 == 0:      # faithful naive engine is slow: sample it
            t0 = time.perf_counter(); mn = YappuDatalog(GRAMMAR_CLAUSES, facts); DL['t_naive'].append(time.perf_counter() - t0)
            DL['naive_checked'] += 1; DL['naive_fact_equal'] += set(mn.F) == set(mi.F)
            if len(js_cases) < 12 and (op_dl or any(r['lof'] for r in raw_py.values())): js_cases.append((facts, set(mi.F)))
_b = np.zeros((GH, GW), bool); _b[SEED_CELL[1], SEED_CELL[0] + 1] = True
assert not valid_path([SEED_CELL, (SEED_CELL[0] + 1, SEED_CELL[1])], _b)          # steps onto a barrier
assert not valid_path([SEED_CELL, (SEED_CELL[0] + 1, SEED_CELL[1] + 1)], np.zeros((GH, GW), bool))   # diagonal step
ms = lambda v: round(1000 * float(np.median(v)), 3)
results['R5_yappu_datalog_backend'] = dict(
    rules_as_sentences=[t for _, t in GRAMMAR_RULES], compiled=[c['text'] + f'   % stratum {c["stratum"]}' for c in GRAMMAR_CLAUSES],
    heldout_frames=DL['frames'], agreement_perimeter=DL['agree_perim'] / DL['frames'],
    agreement_line_of_fire=DL['agree_lof'] / max(1, DL['n_workers']), agreement_red_zone=DL['agree_red'] / max(1, DL['n_workers']),
    median_ms_python_rules=ms(DL['t_py']), median_ms_indexed_datalog=ms(DL['t_ix']), median_ms_faithful_yappu_engine=ms(DL['t_naive']),
    p95_ms_indexed_datalog=round(1000 * float(np.percentile(DL['t_ix'], 95)), 3),
    faithful_vs_indexed_identical_fixpoint=f"{DL['naive_fact_equal']}/{DL['naive_checked']}",
    breach_proof_paths_valid=f"{DL['path_valid']}/{DL['path_n']}",
    gap_from_proof_path_err_m_median=float(np.median(DL['loc_err'])) if DL['loc_err'] else None)
node = shutil.which('node')
if node and js_cases:
    os.makedirs('vg360_js', exist_ok=True)
    open('vg360_js/rj_check.js', 'w').write(RJ_JS)
    json.dump(dict(clauses=GRAMMAR_CLAUSES, cases=[[list(f) for f in c] for c, _ in js_cases]), open('vg360_js/cases.json', 'w'))
    r = subprocess.run([node, 'vg360_js/rj_check.js', 'vg360_js/cases.json'], capture_output=True, text=True)
    js = json.loads(r.stdout)
    same = sum(sorted('|'.join(map(str, f)) for f in py) == j for (_, py), j in zip(js_cases, js))
    results['R5_yappu_datalog_backend']['matches_yappu_js_engine'] = f'{same}/{len(js_cases)} frames, fact-for-fact'
else:
    results['R5_yappu_datalog_backend']['matches_yappu_js_engine'] = 'skipped (node not found)'

# ---------------------------------------------------------------- 10c. camera drift: knocked camera, covered lens
def drift_run(ep, bump, monitor_on):
    eng = SafetyGrammarEngine(); Track._n = 0; rr = np.random.default_rng(ep); mon = None
    rows = []
    for t, dets, gt in sim_episode(1000 + ep, T=T_EP, bump=bump):
        if t == 0 and monitor_on: mon = DriftMonitor(render_frame(rr), CALIB)    # reference frame at calibration time
        est = mon.check(render_frame(rr, dets, gt['blind'])) if (mon is not None and t % DRIFT_EVERY == 0) else None
        rep = eng.process(dets, drift_est=est)
        own = {}
        for tr in eng.tracks:
            if tr.votes and t - tr.last <= 15:
                wid, _ = tr.votes.most_common(1)[0]
                if isinstance(wid, int) and (wid not in own or tr.last > own[wid].last): own[wid] = tr
        wmap = {w['track']: w for w in rep['workers']}
        for wid, g in gt['workers'].items():
            tr = own.get(wid); w = wmap.get(tr.id) if tr is not None else None
            lof = w['line_of_fire'] if w else False; red = w['red_zone'] if w else False
            if rep['calibration']['state'] == CAL_UNKNOWN: lof = red = UNK      # calibration lost: nothing is decided
            rows.append(dict(kind='w', t=t, after=gt['bumped'], blind=gt['blind'], gt_lof=g['lof'], lof=lof, gt_red=g['red'], red=red))
        if not rep['perimeter']['warming_up']:
            rows.append(dict(kind='p', t=t, after=gt['bumped'], blind=gt['blind'], gt=gt['breach'], breach=rep['perimeter']['breach'],
                             state=rep['calibration']['state']))
    return rows

def _decided_mcc(rows, gk, pk):
    d = [r for r in rows if r[pk] not in (UNK, None)]
    return (mcc([r[gk] for r in d], [bool(r[pk]) for r in d]) if d else None), 1 - len(d) / max(1, len(rows))

DRIFT = {}
for scen in ('nominal', 'knocked_3deg', 'lens_covered'):
    for mon in (False, True):
        W, P, delays = [], [], []
        for ep in SPLIT['test']:
            r = np.random.default_rng(ep + 77)
            bump = None if scen == 'nominal' else (dict(t=int(r.integers(60, 150)), tilt=3.0) if scen == 'knocked_3deg' else dict(t=int(r.integers(100, 180)), blind=True))
            rows = drift_run(ep, bump, mon)
            sel = (lambda x: True) if scen == 'nominal' else (lambda x: x['after'])
            W += [x for x in rows if x['kind'] == 'w' and sel(x)]; P += [x for x in rows if x['kind'] == 'p' and sel(x)]
            if bump is not None and mon:
                tb = bump['t']; hit = next((x['t'] for x in rows if x['kind'] == 'p' and x['t'] >= tb and x['state'] != CAL_OK), None)
                if hit is not None: delays.append(hit - tb)
        m_lof, u_lof = _decided_mcc(W, 'gt_lof', 'lof'); m_red, _ = _decided_mcc(W, 'gt_red', 'red'); m_p, u_p = _decided_mcc(P, 'gt', 'breach')
        key = f"{scen}|monitor_{'on' if mon else 'off'}"
        DRIFT[key] = dict(frames_scored=len(P), MCC_line_of_fire=m_lof, MCC_red_zone=m_red, MCC_perimeter=m_p,
                          unknown_rate=round(u_p, 3),
                          states={k: sum(1 for x in P if x['state'] == k) for k in (CAL_OK, CAL_CORRECTED, CAL_UNKNOWN)},
                          detect_delay_frames_median=float(np.median(delays)) if delays else None)
        if scen == 'lens_covered':
            DRIFT[key]['decided_while_blind'] = round(1 - u_p, 3)
            DRIFT[key]['worker_verdicts_decided_while_blind'] = round(1 - u_lof, 3)
            for k in ('MCC_line_of_fire', 'MCC_red_zone', 'MCC_perimeter'): DRIFT[key].pop(k)   # no truth to score while blind
            DRIFT[key]['false_breach_alarms_while_blind'] = sum(1 for x in P if x['breach'] is True and not x['gt'])
        print(f"[drift] {key:30s} " + json.dumps(sanitize(DRIFT[key])), flush=True)
results['R6_camera_drift'] = DRIFT
results['R0_lens'] = dict(camera=CAMERA_PROFILE, method=LENS_METHOD, report=LENS_REPORT,
                          k_true=DIST_TRUE[:2].tolist(), k_fitted=CALIB['D'][:2].tolist())

# ---------------------------------------------------------------- 10d. golden replay for the Rust port (tamserve-boxes)
EXPORT_RUST_GOLDEN = True
def _r(x, n): return [round(float(v), n) for v in x]
def export_rust_golden(path='vg360_rust_golden.json'):
    def mlp_json(head):
        lin = [m for m in head.model.net if isinstance(m, nn.Linear)]
        return dict(layers=[dict(w=m.weight.detach().numpy().astype(np.float32).tolist(), b=m.bias.detach().numpy().astype(np.float32).tolist()) for m in lin],
                    mu=head.mu.tolist(), sd=head.sd.tolist(), q=[head.q[0], head.q[1]])
    site = dict(floor_w=FLOOR_W, floor_h=FLOOR_H, cell=CELL, r_lof=R_LOF, link_m=LINK_M, post_win=POST_WIN, post_min=POST_MIN,
                post_r=POST_R, s_max=S_MAX, hi=HI, lo=LO, ppe_win=PPE_WIN, guardrails=[[list(a), list(b)] for a, b in GUARDRAIL],
                red_rect=list(RED_RECT), spill_c=list(SPILL_C), ring_half=RING_HALF, hazard_radius=1.0, fill_box=list(FILL_BOX),
                drift_ok_px=DRIFT_OK_PX)
    out = dict(site=site, static_grid=STATIC.tolist(), static_barrier=STATIC_BARRIER.astype(int).tolist(),
               calib=dict(K=CALIB['K'].tolist(), D=CALIB['D'].tolist(), H=CALIB['H'].tolist()),
               ppe={k: mlp_json(h) for k, h in HEADS.items()}, clauses=GRAMMAR_CLAUSES, episodes=[])
    for name, ep, T, bump, brk in (('breach_and_knocked_3deg_at_100', 1030, 240, dict(t=100, tilt=3.0), True),
                                   ('lens_covered_at_120', 1031, 200, dict(t=120, blind=True), None)):
        eng = DatalogSafetyEngine(ppe_heads=HEADS); Track._n = 0; rr = np.random.default_rng(ep); mon = None; frames = []
        for t, dets, gt in sim_episode(ep, T=T, bump=bump, breach=brk):
            if t == 0: mon = DriftMonitor(render_frame(rr), CALIB)
            est = mon.check(render_frame(rr, dets, gt['blind'])) if t % DRIFT_EVERY == 0 else None
            d2 = [dict(cls=d['cls'], conf=round(float(d['conf']), 4), box=_r(d['box'], 2)) for d in dets]
            rep = eng.process(d2, drift_est=est)
            pr = rep['perimeter']
            exp = dict(calib=rep['calibration']['state'], breach=pr['breach'], raw_open=pr['raw_open'], posts=pr['posts_persistent'],
                       gap=pr['gap_floor_xy'], escape_path=pr.get('proof', {}).get('escape_path'),
                       workers=[dict(track=w['track'], cell=list(w['cell']), xy=[float(v) for v in eng_tr_pos(eng, w['track'])],
                                     lof=w['line_of_fire'], red=w['red_zone'], helmet=w['helmet'], vest=w['vest']) for w in rep['workers']])
            frames.append(dict(dets=d2, drift=None if est is None else dict(valid=est['valid'], G=est['G'], drift_px=est['drift_px'],
                                                                           inliers=est['inliers'], ratio=est['ratio']), expected=exp))
        out['episodes'].append(dict(name=name, frames=frames))
    json.dump(sanitize(out), open(path, 'w'), separators=(',', ':'))
    return path, os.path.getsize(path)
def eng_tr_pos(eng, tid): return next(tr.pos for tr in eng.tracks if tr.id == tid)
if EXPORT_RUST_GOLDEN:
    _gp, _gs = export_rust_golden(); results['rust_golden'] = dict(path=_gp, bytes=_gs)
    print(f"[rust] golden replay written: {_gp} ({_gs/1e6:.2f} MB)")

# ---------------------------------------------------------------- 11. OpenAI-compatible endpoint
from fastapi import FastAPI, Request
import uvicorn, httpx
PORT, MODEL_ID = 8765, 'vg360-array-grammar-yappu-v2'
LIVE = DatalogSafetyEngine(ppe_heads=HEADS)   # yappu Datalog back end: every alert carries its proof
_lock = threading.Lock()
app = FastAPI()

@app.get('/v1/models')
def models(): return {'object': 'list', 'data': [{'id': MODEL_ID, 'object': 'model', 'owned_by': 'innoura'}]}

@app.post('/v1/chat/completions')
async def chat(req: Request):
    body = await req.json(); msgs = body.get('messages', [])
    user = next((m.get('content', '') for m in reversed(msgs) if m.get('role') == 'user'), '')
    with _lock:
        try:
            payload = json.loads(user) if isinstance(user, str) else user
            rep = LIVE.process(payload['detections']); content = json.dumps(sanitize(rep))
        except (json.JSONDecodeError, KeyError, TypeError):
            alarms = [w for w in LIVE.tracks if any(w.alarm.values())]
            content = json.dumps(sanitize(dict(frame=LIVE.f, active_alarms=[dict(track=t.id, **{k: v for k, v in t.alarm.items() if v}) for t in alarms],
                                               perimeter_breach=LIVE.perim_alarm,
                                               hint='send {"detections":[{"cls","conf","box":[x1,y1,x2,y2]}]} as the user message')))
    pt, ct = len(str(user).split()), len(content.split())
    return {'id': f'chatcmpl-{int(time.time()*1e6)}', 'object': 'chat.completion', 'created': int(time.time()), 'model': body.get('model', MODEL_ID),
            'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': content}, 'finish_reason': 'stop'}],
            'usage': {'prompt_tokens': pt, 'completion_tokens': ct, 'total_tokens': pt + ct}}

server = uvicorn.Server(uvicorn.Config(app, host='0.0.0.0', port=PORT, log_level='warning'))
def _serve():
    loop = asyncio.new_event_loop(); asyncio.set_event_loop(loop); loop.run_until_complete(server.serve())
threading.Thread(target=_serve, daemon=True).start()
for _ in range(100):
    if server.started: break
    time.sleep(0.05)
last, proof_perim, proof_worker = None, None, None
with httpx.Client(timeout=10) as cli:
    for t, dets, gt in sim_episode(4242, T=260, breach=True, steam=False):
        dets = [{k: v for k, v in d.items() if not k.startswith('_')} for d in dets]   # strip sim-only labels
        r = cli.post(f'http://127.0.0.1:{PORT}/v1/chat/completions',
                     json={'model': MODEL_ID, 'messages': [{'role': 'user', 'content': json.dumps({'detections': dets})}]})
        last = r.json(); rep = json.loads(last['choices'][0]['message']['content'])
        if proof_perim is None and 'proof' in rep['perimeter']: proof_perim = rep['perimeter']['proof']
        if proof_worker is None:
            proof_worker = next((w['proof'] for w in rep['workers'] if 'proof' in w), None)
    status = cli.post(f'http://127.0.0.1:{PORT}/v1/chat/completions', json={'model': MODEL_ID, 'messages': [{'role': 'user', 'content': 'status'}]}).json()
results['endpoint'] = dict(url=f'http://127.0.0.1:{PORT}/v1/chat/completions', last_frame_reply=json.loads(last['choices'][0]['message']['content']),
                           example_perimeter_proof=proof_perim, example_worker_proof=proof_worker,
                           status_reply=json.loads(status['choices'][0]['message']['content']))

# ---------------------------------------------------------------- 12. report + one visual
results['config'] = dict(grid=f'{GW}x{GH} @ {CELL} m', R_LOF_m=R_LOF, hysteresis=dict(S=S_MAX, HI=HI, LO=LO, min_dwell_frames=HI - LO),
                         homography_err_cm=round(HOMO_ERR_M * 100, 1), episodes=N_EP, frames_per_episode=T_EP, runtime_s=round(time.time() - T0, 1))
print(json.dumps(sanitize({k: v for k, v in results.items() if k != 'endpoint'}), indent=1))
print('[endpoint] last frame reply (truncated):', json.dumps(sanitize(results['endpoint']['last_frame_reply']))[:600])
print('[endpoint] example worker proof:\n  ' + '\n  '.join(results['endpoint']['example_worker_proof'] or ['(none)']))
_pp = results['endpoint']['example_perimeter_proof']
if _pp: print('[endpoint] example breach proof: rule:', _pp['rule'], '| path cells:', len(_pp['escape_path']), '| valid:', _pp['path_valid'], '| gap at', _pp['gap_floor_xy'], '\n  path:', _pp['escape_path'])
print('[endpoint] status reply:', json.dumps(sanitize(results['endpoint']['status_reply'])))
json.dump(sanitize(results), open('vg360_grammar_results.json', 'w'), indent=1)
try:
    import matplotlib; matplotlib.use('Agg') if not IN_COLAB else None
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap
    eng = SafetyGrammarEngine(ppe_heads=HEADS); Track._n = 0; snap = None
    for t, dets, gt in sim_episode(77, T=260, breach=True, steam=False):
        rep = eng.process(dets)
        if rep['perimeter']['breach'] and any(w['line_of_fire'] for w in rep['workers']): snap = (dets, rep, eng.grid.copy()); break
        snap = (dets, rep, eng.grid.copy())
    dets, rep, grid = snap
    fig, ax = plt.subplots(1, 2, figsize=(14, 5.2))
    ax[0].set_xlim(0, IMG_W); ax[0].set_ylim(IMG_H, 0); ax[0].set_facecolor('#222'); ax[0].set_title(f'camera view (YOLOX-like boxes), frame {rep["frame"]}')
    col = dict(person='#4fc3f7', helmet='#ffd54f', vest='#ff8a65', forklift='#e57373', barrier_post='#aed581')
    for d in dets:
        x1, y1, x2, y2 = d['box']; ax[0].add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec=col[d['cls']], lw=1))
    cmap = ListedColormap(['#f5f5f5', '#ef9a9a', '#455a64', '#ffcc80', '#1e88e5', '#c62828'])
    ax[1].imshow(grid, cmap=cmap, vmin=0, vmax=5, origin='lower', extent=(0, FLOOR_W, 0, FLOOR_H))
    for w in rep['workers']:
        c = 'red' if (w['line_of_fire'] or w['red_zone']) else 'black'
        ax[1].annotate(f"T{w['track']} H:{w['helmet'][0]} V:{w['vest'][0]}", w['floor_xy'], color=c, fontsize=7, xytext=(3, 3), textcoords='offset points')
    if rep['perimeter']['gap_floor_xy']: ax[1].plot(*rep['perimeter']['gap_floor_xy'], 'mx', ms=14, mew=3)
    ax[1].set_title(f"floor token grid: breach={rep['perimeter']['breach']}  (x = localised gap; red label = LOF/red-zone alarm)")
    plt.tight_layout(); plt.savefig('vg360_grammar_frame.png', dpi=110)
    if IN_COLAB: plt.show()
except Exception as e:
    print('[viz] skipped:', e)
print(f'[done] {time.time()-T0:.1f}s')
